# 16. The attribution of the factor objective's paths: what the budget bought, factor by factor

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same period |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Cap-weighted | weighted by market capitalisation |
| Benchmark | in the extension, the index the portfolio tracks, the cap-weighted combination of the 49 industries built in notebook 08 |
| Variance | the average squared distance of a series from its own average; its square root is the standard deviation, the usual measure of how much a return moves; volatility is the standard deviation of returns, stated per year here by multiplying the monthly figure by the square root of 12 |
| Standard error | the uncertainty of a number estimated from a sample: the standard deviation that the estimate would show across repeated samples of the same size; for a mean it is the standard deviation of the observations divided by the square root of their number |
| Covariance matrix | the table of all variances and covariances of a set of assets, 49 by 49 here |
| Covariance estimator | a method for estimating the covariance matrix from a window of data; notebook 10 built four, and notebook 12 ran them with the robust portfolio and RiskMetrics as seven variants |
| Estimation window | the past returns an estimator sees, 120 months or three years of trading days here |
| Shrinkage | pulling an estimate part of the way towards a simpler target, which trades a little bias for less estimation error |
| RiskMetrics | the covariance estimator of J.P. Morgan's 1996 RiskMetrics document: a weighted sample covariance in which each day's weight falls by half every half-life, one year of trading days here, so that recent days count most (notebook 10's sensitivity) |
| Factor | a return series that moves many assets at once; the six here are the market factor Mkt-RF and five long-short portfolios: SMB (small firms over large, size), HML (cheap firms over expensive, value), RMW (profitable firms over unprofitable, profitability), CMA (firms that invest little over firms that invest much, investment) and Mom (recent winners over recent losers, momentum) |
| Premium | the average return of a factor over long samples, as the literature reports it; the four factors value, profitability, investment and momentum have a documented positive premium, and this study estimates none of them |
| Principal component | a combination of the assets, with one weight per asset, chosen so that it explains as much of the assets' total variance as a single combination can (notebook 09); the estimator named after it builds the covariance from five of them (notebook 10) |
| Beta | how much an asset moves with a factor on average, estimated by regressing the asset's excess returns on the factors' returns over the estimation window; one beta per factor; the beta to the benchmark is how much it moves with the benchmark, one for the benchmark itself |
| Active weight | the portfolio's weight in an asset minus the benchmark's weight in it |
| Active return | the portfolio's return minus the benchmark's return in the same month |
| Active exposure | the portfolio's beta to a factor minus the benchmark's: the sum over industries of the active weight times the industry's beta to that factor (notebook 13) |
| Contribution | the part of a month's active return that one factor accounts for: the active exposure to it, with the betas estimated before the month, times the factor's return in the month |
| Residual | the active return minus the six contributions, the part the industries' own returns account for |
| Identity | the check that the six contributions plus the residual equal the active return in every month, to the arithmetic's precision |
| Factor share | the share of the variance of a path's active return that the six contributions explain: one minus the variance of the residual over the variance of the active return |
| Variance decomposition | the active return's variance split into three shares that sum to one: the factor total's own variance, the residual's own variance and twice the covariance of the two, each over the active return's variance; the factor share is one minus the residual's share, and the covariance share is negative when the two parts move against each other |
| Attribution | splitting a portfolio's active return into the contributions of named factors plus a residual, so that the parts add up to the whole (notebook 13) |
| Verdict | the strict pass or fail of an expectation, MET or NOT MET, as the design requires |
| Tracking error | the standard deviation of the difference between two return series, stated per year; forecast (ex-ante) when computed from a covariance matrix before the month, realised (ex-post) when measured on the active returns afterwards |
| Long-only | said of a portfolio in which no weight is negative |
| One-way turnover | half the sum over industries of the absolute changes in weight at a rebalance; the fraction of the portfolio sold, which is also the fraction bought when the portfolio stays fully invested |
| Turnover cap | an upper limit on the one-way turnover of a rebalance; 2% a month here |
| Drifted weights | the previous month's weights after that month's returns have moved them, so that they still sum to one |
| Tilt | a constraint that holds named industries below their benchmark weight; here Coal, Oil and Utilities at no more than half of it. The word keeps this one meaning in version 2 |
| Exclusion | a constraint that holds named industries at zero weight; Smoke (tobacco) and Guns (weapons) here |
| Beta-neutral | a constraint that holds the portfolio's beta to the benchmark equal to one |
| Mandate | the whole set of rules the fund obeys: the tilt, the exclusions and beta neutrality, with the active weight bound and the turnover cap, the constraint set C3 of version 1 |
| Optimiser | the extension's routine that solves a constrained portfolio problem; version 1's minimises the forecast tracking error (notebook 11), version 2's maximises the factor objective (notebook 14) |
| Factor objective | the quantity version 2's optimiser maximises: the sum of the portfolio's active exposures to the targeted factors, each counted in the direction of its premium |
| Targeted factors | the four of the six factors with a documented premium, value, profitability, investment and momentum, each targeted in the direction of its premium and weighted equally in beta units; the two single-factor paths target one of them alone |
| Budget | the limit on the forecast tracking error of the factor objective's portfolio: 75, 100 or 150 basis points a year. A month in which the mandate's tracking-error-minimising portfolio already exceeds the budget takes that portfolio and is counted |
| Main path | one of the fifteen paths of the five covariance estimators at the three budgets; the robust, RiskMetrics and single-factor paths are sensitivities |
| Exposure bought | the mean active exposure a path carries per basis point of the tracking error it realised; per factor, the mean active exposure to that factor over the realised tracking error in basis points a year |
| Gap | for a month, the factor objective the optimiser chose with the betas estimated before the month, minus the value the same active weights carry with the betas estimated after it, s'(B_t minus B_t+1)' a_t; its mean absolute value over the months is the path's gap |
| Gap per unit | the path's gap divided by its mean factor objective: the part of the exposure bought that the next month's betas take back, as a fraction |
| Robust portfolio | the weights whose largest forecast tracking error across the five covariance estimates is smallest (notebook 11); under the factor objective, the weights whose budget holds under all five at once |
| Path | the month-by-month sequence of portfolios of one specification, each built on the data before its month; here one variant at one budget with one target vector |
| Quadratic program | an optimisation whose objective is quadratic and whose constraints are linear, which the solver reaches exactly; a cone program adds constraints written as cones, which an interior-point method reaches to a set precision. The factor objective's problem is a cone program because of the budget |
| Basis point | one hundredth of a percentage point, so 100 basis points is 1.00% |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage; and, for a run, the solver's name and version |

**The problem, stated the way practitioners state it.** A manager who spent a budget on the forecast tracking error on exposures to four factors owes the client an account of where the active return came from: how much each factor contributed, how much came from the industries' own returns beside the factors, and how much of the month-to-month movement of the active return the factors explain. The same account tells the manager what the budget bought, factor by factor, per basis point of the risk it cost, and how much of the exposure chosen on the betas of the past the betas of the following month take back. This notebook gives that account for the nineteen paths of notebook 15.

**What this notebook does.** It is the third of three (14 the problem, 15 the rolling evaluation, 16 the attribution). It rebuilds the nineteen paths from the modules with their weights kept, as notebook 13 rebuilds version 1's, checks their realised tracking errors against notebook 15's within 0.1 basis point a year and, where the repository is present, their monthly records against the committed table under the rule for reruns; it attributes each path's active return exactly as notebook 13 does, into the six factors' contributions and a residual, with the identity checked in every month; it judges E4, the one expectation left to it, and reports the variance decomposition behind the factor share; and it reports two measures in no claim, the exposure bought per factor and the gap per unit of exposure bought, both named in `constants.py` on 8 November 2026 before this code.

**The expectation (`constants.py`, 8 October 2026, before any path ran).**

- E4. The four targeted factors' contributions sum to a positive amount a year on every main path, and at 100 and 150 basis points the six factors explain at least 30% of the active return's variance on every main path. Reason: version 1's factors explained 14% to 28% of incidental exposures; deliberate exposures raise the share.

**The two measures, reported with no expectation attached.**

- The exposure bought per factor: the mean active exposure to each targeted factor divided by the path's realised tracking error in basis points a year; the four sum to notebook 15's exposure per basis point.
- The gap per unit: the path's mean absolute gap divided by its mean factor objective. Notebook 15 recorded the gap month by month; this notebook tabulates it for every path and states it per unit of what was bought.

**The checks that are not expectations.** The rebuilt paths reproduce notebook 15's realised tracking errors within 0.1 basis point (holds or fails); where the committed monthly table is present, the rule for reruns is reported: the records of a cone program within 1e-8 per cell, the inaccurate months counted beside and not part of it; the identity holds within 1e-9 in every month (holds or fails).

Running time: about ten minutes, most of it the 10,754 cone programs of the nineteen paths.

## Modules

`constants` carries section 7 with E4 and the two measures of this notebook; `attribution` is notebook 13's, unchanged; `evaluation`'s `run_factor_path` is notebook 15's; the others are those of earlier notebooks.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("cvxpy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cvxpy"])
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
# The end of the sample, fixed on 6 November 2026. Until then the end was the
# latest month the vintage carried, and every recorded run (7 to 18 October
# 2026) used the 202608 vintage, whose last month is 2026-08, the end the design
# names ("1979-07 to 2026-08"). French's library moved to the 202609 vintage
# before 6 November, which carries 2026-09 and so gave 567 evaluation months
# and stopped every notebook at its count. From now on the loader cuts every
# file at this month, whatever the vintage, so that each notebook sees the same
# months on any later vintage; the vintage line and the SHA-256 in the
# provenance record say which revision of French's history a run used, and a
# later vintage need not reproduce every digit of the committed tables, since
# French revises history. A run on the cached 202608 files is unchanged by the
# cut. Not an expectation: a data rule, disclosed here and in the README.
EXT_SAMPLE_END = "2026-08"
# The rule in the design's words, written here on 8 November 2026 with the
# design's approval of 6 November, and carried in the README's run section in
# the same words: the sample is July 1979 to August 2026, fixed in the code.
# The recorded tables are those of the 202608 vintage as cached. A run on a
# later vintage, cut at August 2026, is a confirmation run: it is labelled by
# its vintage in the provenance record, it is expected to reproduce every
# verdict line and every printed decimal and not every digit, because French
# revises history, and it replaces no committed table without a dated decision.
EXT_SAMPLE_END_RULE_DATE = "2026-11-08"
EXT_CONFIRMATION_RUN = "a run on a later vintage, cut at August 2026, labelled by its vintage in the provenance record"

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators
# Additions of 18 October 2026 to notebook 12, by the design of 9 October 2026,
# fixed here before their code, after the recorded run of 7 October (commit
# 6835d49) and labelled in the notebook. (a) Two saves, so that the dashboard
# can draw the forecasts and the holdings from committed tables: the monthly
# forecast tracking error of every path, beside the active returns the
# notebook already writes, and the weights every path holds in the last
# evaluation month, with the benchmark's. (b) Two descriptive figures per path,
# reported beside the standard-error sentence and entering no claim: the worst
# cumulative fall of the active return (the largest fall of the cumulative
# active return from an earlier high, with its peak and trough months) and the
# share of months with a positive active return. (c) A 95 per cent band on the
# bias statistic from a circular block bootstrap of the monthly pairs (active
# return, forecast) in blocks of EVAL_BOOTSTRAP_BLOCK_MONTHS, with
# EVAL_BOOTSTRAP_DRAWS resamples from the seed below; a block keeps the months'
# order, so the band allows for the clustering of the active returns, and the
# circle lets a block run past the last month into the first. No normal band is
# drawn. The one expectation attached, expectation 10: after the re-run every
# table the notebook wrote on 7 October (EVAL_RERUN_IDENTICAL, the measures,
# the by-decade table, the calibration table and the active returns) comes
# out byte for byte equal to the committed copy in results/<vintage>/, so that
# the additions change no number of version 1.
EVAL_ADDITIONS_DATE = "2026-10-18"
EVAL_BOOTSTRAP_BLOCK_MONTHS = 12
EVAL_BOOTSTRAP_DRAWS = 2000
EVAL_BOOTSTRAP_SEED = 20261009               # the date of the design's ruling, as a number
EVAL_BOOTSTRAP_LEVEL = 0.95
EVAL_RERUN_IDENTICAL = ("rolling_evaluation", "rolling_evaluation_by_decade", "rolling_evaluation_calibration", "active_returns")   # expectation 10

# Additions of 6 November 2026 to notebook 12, after the runs of 7 and 18
# October. (a) The rolling realised tracking error of every path: the standard
# deviation (n minus 1, as in realised_tracking_error) of the monthly active
# return over the EVAL_ROLLING_TE_MONTHS months ending in and including each
# month, times the square root of 12, from the first month with a full window;
# and the median across the seven variants under C3 and under the tilt alone.
# A reported measure in no claim, with no expectation attached; the window is
# three years, the length of the daily covariance windows. Written as a table
# so that a figure drawn from the series is traced to a committed table like
# every other figure. (b) The rule for every rerun from now on, set by the
# design on 6 November 2026 beside expectation 10 and not part of its verdict,
# which stays as written: a table from the quadratic programs reproduces the
# committed copy to the last digit; a table from the cone program (the robust
# variant, ROBUST_VARIANT) reproduces it to EVAL_RERUN_CONE_MAX_DIFFERENCE per cell;
# the solver's name and version are written to the run's provenance record
# (provenance_solver.json); the count of months with an inaccurate solver
# status is reported beside the comparison and is not part of it.
EVAL_ROLLING_ADDITIONS_DATE = "2026-11-06"
EVAL_ROLLING_TE_MONTHS = 36
EVAL_RERUN_CONE_MAX_DIFFERENCE = 1e-8        # the monthly records of the cone program (active_returns)
# The measures' own number, set by the design on 7 November 2026 after the run
# of 6 November showed the robust variant's by-decade and calibration rows up
# to 3e-8 and 2e-8 from their committed copies while its monthly active returns
# were within 3.2e-9: a measure computed over the 566 months from those
# records, a standard deviation or a ratio, moves by a multiple of what its
# inputs move by, so a table of measures from the cone program (the three
# measure tables of version 1 and the rolling tracking-error table) reproduces
# its committed copy to EVAL_RERUN_CONE_MEASURES_MAX_DIFFERENCE per cell. The
# records' number above stays as written; the verdict of expectation 10 is
# unchanged.
EVAL_RERUN_CONE_MEASURES_MAX_DIFFERENCE = 1e-7
EVAL_RERUN_MEASURE_TABLES = ("rolling_evaluation", "rolling_evaluation_by_decade", "rolling_evaluation_calibration", "rolling_tracking_error", "rolling_evaluation_bias_by_decade")   # the last added 8 November 2026

# Notebook 13, the factor attribution, fixed 6 October 2026 before its code: each
# path's monthly active return split into the contributions of the six factors
# (the active exposure to each, from the industries' betas estimated on the data
# before the month, times the factor's return in the month) plus a residual, the
# industries' own returns; the identity is the check. The paths attributed are
# the seven variants under the mandate's C3 and under the tilt alone, rebuilt
# from the modules; the expectations are in the notebook's introduction.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # expectation 1: contributions plus residual minus active return, per month, in return units
ATTRIBUTION_SETS = ("C3", "C3 tilt only")        # the two constraint sets whose paths are attributed
ATTRIBUTION_REPRODUCTION_TOL = 0.00001           # expectation 1: the rebuilt paths reproduce notebook 12's realised tracking errors within 0.1 basis point a year
ATTRIBUTION_REFERENCE_TE = {                     # notebook 12's realised tracking errors, per year, for the reproduction check
    ("sample, daily 3y", "C3"): 0.0084654, ("ledoit_wolf, daily 3y", "C3"): 0.0084820, ("factor, daily 3y", "C3"): 0.0093449,
    ("pca, daily 3y", "C3"): 0.0091482, ("sample, monthly 120", "C3"): 0.0095761, ("robust, all five", "C3"): 0.0087358,
    ("RiskMetrics, daily 5y", "C3"): 0.0084278,
    ("sample, daily 3y", "C3 tilt only"): 0.0080276, ("ledoit_wolf, daily 3y", "C3 tilt only"): 0.0080282, ("factor, daily 3y", "C3 tilt only"): 0.0088296,
    ("pca, daily 3y", "C3 tilt only"): 0.0086722, ("sample, monthly 120", "C3 tilt only"): 0.0090924, ("robust, all five", "C3 tilt only"): 0.0082305,
    ("RiskMetrics, daily 5y", "C3 tilt only"): 0.0079698,
}
ATTRIBUTION_MARKET_EXPOSURE_MAX = 0.05           # expectation 2: the mean active exposure to Mkt-RF, in absolute value, for every path
ATTRIBUTION_SIGN_AGREEMENT_MIN = 4               # expectation 3: of the five factors other than the market, the number whose mean active exposure under the mandate has the same sign for all seven variants
ATTRIBUTION_FACTOR_SHARE_MAX = 0.5               # expectation 4: the share of the active return's variance the six factors explain, for every path
ATTRIBUTION_FACTOR_PART_MAX = 0.5                # expectation 5a: the share of the mandate's cost against the tilt alone that comes through the factor contributions, every variant
ATTRIBUTION_ARITHMETIC_TOL = 0.0002              # expectation 5b: the exclusions' factor part (the five factors other than the market) within this of the arithmetic of the inputs, per year (2 basis points), every variant

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 7. Version 2, module A: the factor objective under the mandate
# ---------------------------------------------------------------------------
# Fixed 8 October 2026, before any of the module's code was written, from the
# design of 7 October 2026 and version 1's results at commit 6835d49. Version 1 found that
# the mandate's tilt leaves incidental exposures of 0.004 to 0.018 in absolute
# value to the six factors, costing 9 to 13 basis points a year. Module A makes
# the exposures the objective: each month, with the data available before the
# month, the optimiser maximises the sum of the active exposures to the four
# targeted factors, s' B' (w - b), where B holds the industries' betas to the
# six factors estimated as in notebook 13 (three years of daily returns before
# the month) and s is FO_TARGET_SIGN, under every constraint of the mandate
# (CONSTRAINT_SETS["C3"]) and one new constraint, the budget: the forecast
# tracking error from the month's covariance estimate at most FO_BUDGETS_ANNUAL.
# The study holds no view on expected returns, as version 1 did not: the
# targets are the four of French's six factors with a documented premium, the
# direction of each is the literature's, and the four are weighted equally in
# beta units. Nothing estimates a premium or times a factor.
# Naming, fixed in the design: "tilt" stays the industry reduction of version 1;
# the new objective is the factor objective, the four are the targeted factors,
# the limit is the budget.
FO_DECISION_DATE = "2026-10-08"
FO_TARGET_FACTORS = ("HML", "RMW", "CMA", "Mom")              # value, profitability, investment, momentum
FO_TARGET_SIGN = {"Mkt-RF": 0.0, "SMB": 0.0, "HML": 1.0, "RMW": 1.0, "CMA": 1.0, "Mom": 1.0}   # s, on ATTRIBUTION_FACTORS; +1 is the direction of the documented premium
FO_CONSTRAINT_SET = "C3"                                       # the mandate: every constraint of version 1's C3, unchanged
FO_BUDGETS_ANNUAL = (0.0075, 0.0100, 0.0150)                   # tau, per year: just above the mandate's own realised cost (84.7 to 95.8 bp), the enhanced-indexing limit of OPT_TILT_TE_MAX_ANNUAL, and the generous case
FO_BUDGET_MAIN = 0.0100                                        # the budget at which the robust, RiskMetrics and single-factor paths run and most expectations are tested
FO_BETA_WINDOW_YEARS = EXT_COV_DAILY_WINDOW_YEARS              # the betas' window: notebook 13's three years of daily returns before the month
# Infeasibility rule: in a month in which the budget cannot be met under the
# mandate (the tracking-error-minimising solution of version 1 already exceeds
# tau), the path takes that minimising solution, the month is counted and
# reported, and nothing else is relaxed; the count is one of the results.
FO_INFEASIBLE_RULE = "tracking-error minimiser"
# The 19 paths, the count fixed here: the five estimator variants of version 1
# at each of the three budgets (15); the robust and RiskMetrics variants at the
# main budget (2); two single-factor sensitivities at the main budget with the
# daily sample estimator, s on value alone and on momentum alone (2). Version
# 1's mandate paths (tracking error minimised, no objective) are the comparison
# and are not rerun. A path added after the first run is disclosed with its
# date and reason, and the count restated.
FO_SINGLE_FACTOR_PATHS = (("HML",), ("Mom",))
FO_PATH_COUNT = 19
# The two measures new to module A. The exposure the budget bought: the mean
# over the months of the sum of the four targeted active exposures, divided by
# the realised tracking error in basis points a year. The gap: for each month,
# the targeted exposure the optimiser chose ex ante, s' B_t' a_t with B_t the
# betas estimated before the month, minus the exposure the same weights carry
# when the betas are estimated after the month, s' B_{t+1}' a_t; its mean
# absolute value over the months is the path's gap.
FO_GAP_DEFINITION = "s'(B_t - B_{t+1})' a_t, mean absolute value over the months"
# The seven expectations, each a strict pass or fail; the levels from
# version 1's results at 6835d49, with the design's reasons.
# E1 and E2 were restated on 9 October 2026, after notebook 14's check of
# the arithmetic and before any path ran, by the design of 9 October 2026:
# a level changed at that point, between the pre-run check notebook 14
# exists for and the first rolling run, is a pre-run restatement, dated and
# disclosed here and once in the write-up; a level changed after notebook 15
# runs is a post-run change. The original levels of 8 October stay below,
# marked as superseded.
FO_RESTATEMENT_DATE = "2026-10-09"
# E1. At the main budget the budget binds (forecast tracking error within
# FO_E1_BIND_TOL of the budget) in at least FO_E1_BIND_SHARE_MIN of the
# months in which the mandate's own minimum forecast tracking error, with
# turnover free, is at or below the budget, for each of the four daily
# estimators. The months the mandate alone cannot meet are reported per
# estimator and budget from notebook 14's feasibility table; the months the
# path misses beyond them, with the turnover cap in force, are counted and
# reported, since that difference is what the path adds. Reason for the
# restatement: notebook 14 found the mandate alone above 100 bp in 37
# (sample), 43 (Ledoit-Wolf) and 39 (six-factor) of the 566 months, 22 for
# principal components, nearly all in 1979 to 1989, so the budget cannot
# bind in those months whatever the objective.
FO_E1_BIND_TOL_ANNUAL = 0.0001           # one basis point a year
FO_E1_BIND_SHARE_MIN = 0.95
FO_E1_DENOMINATOR = "months the mandate alone can meet"
# Superseded on 9 October 2026: the same share of all 566 months. Reason
# given on 8 October: version 1's forecast under the mandate was 55 to 68 bp
# on average and 47 to 55 for the tilt alone at the 1979 oil peak for the
# daily estimators, so a budget of 100 should be reachable almost always;
# the monthly estimator (108 at that peak) may fail it in the first years.
FO_E1_DENOMINATOR_SUPERSEDED_2026_10_08 = "all 566 months"
# E2. At the main budget the mean active exposure to each targeted factor is
# positive on every main path, the average of the four is at least
# FO_E2_MEAN_EXPOSURE_MIN for the daily sample estimator, and that average
# rises with the budget (75, then 100, then 150) on every estimator; the four
# exposures are reported per path at every budget, the average never alone.
# Reason for the restatement: notebook 14's stationary mean of the four at
# 100 bp is 0.030 for the daily sample estimator and a capped path holds at
# most the stationary exposure on average; the level is four fifths of
# 0.030, the fifth being the design's allowance for what the turnover cap,
# the infeasible months and the rolling betas may cost against the
# stationary solution, a judgement stated as such. Under an active weight
# bound of 2 percentage points on 49 industries, 100 bp of budget buys about
# 0.03 of average exposure in beta units.
FO_E2_MEAN_EXPOSURE_MIN = 0.024
# Superseded on 9 October 2026. Reason given on 8 October: version 1's
# incidental exposures were 0.004 to 0.018 in absolute value with no
# objective; 49 industries within 2 percentage points of their weights can
# reach three to ten times that when the objective asks for it.
FO_E2_MEAN_EXPOSURE_MIN_SUPERSEDED_2026_10_08 = 0.05
# One diagnostic per path, counted and reported with no expectation
# attached (the design of 9 October 2026): the months in which any targeted
# exposure ends below its incidental value, the value the mandate's
# tracking-error minimiser carries. Notebook 14 found such a month in 5 of
# its 15 month-estimator pairs at 100 bp.
FO_DIAGNOSTIC_BELOW_INCIDENTAL = "months with a targeted exposure below its incidental value"
# E3. Realised tracking error exceeds the budget by 20 to 60 per cent on the
# daily estimators, and by more on the monthly sample covariance; tested at
# every budget (the design names none), the monthly ratio against the largest
# daily ratio at the same budget. Reason: version 1's bias was 1.22 to 1.39 on
# daily data and 1.61 on monthly; a budget-binding solution concentrates risk
# where the estimator sees least, so the miss may exceed version 1's, bounded
# here at 1.6.
FO_E3_REALISED_OVER_BUDGET_RANGE = (1.2, 1.6)
# E4. The four targeted factors' contributions sum to a positive amount on
# every main path, and at the main budget and above the six factors explain at
# least FO_E4_FACTOR_SHARE_MIN of the active return's variance on every main
# path. Reason: version 1's factors explained 14 to 28 per cent of incidental
# exposures; deliberate exposures raise the share.
FO_E4_FACTOR_SHARE_MIN = 0.30
# E5. At the main budget the gross active return is positive for at least
# FO_E5_POSITIVE_MIN of the five estimators; the net return at DGU_TCOST per
# unit of turnover lies within FO_E5_NET_GAP_MAX of the gross. Whether the
# active return is more than two standard errors from zero is reported, not
# expected. Reason: the study holds no view on the premia; the sign follows
# from the targets if the premia had their documented sign over the sample,
# which is what the data say and not a claim of skill.
FO_E5_POSITIVE_MIN = 4
FO_E5_NET_GAP_MAX = 0.0010               # 10 basis points a year
# E6. Of the two single-factor paths, momentum alone has the higher turnover
# and the larger gap, and the turnover cap binds in more months on the
# momentum path than on the value path. Reason: industry momentum betas rotate
# within a year; value betas do not.
# E7. At the main budget the spread across the four daily estimators in
# realised tracking error is at most FO_E7_TE_RANGE_MAX, the exposure bought
# per basis point differs across them by less than FO_E7_EXPOSURE_PER_BP_REL_MAX
# of their mean (largest minus smallest over the mean), and the monthly sample
# covariance remains the worst calibrated (the largest realised over forecast
# of the five). Reason: version 1, a range of 9 basis points on a level of
# about 90, the monthly covariance 61 per cent under-forecast.
FO_E7_TE_RANGE_MAX = 0.0020              # 20 basis points a year
FO_E7_EXPOSURE_PER_BP_REL_MAX = 0.30
# Notebook 14 runs the objective at the report months of version 1 and over
# the sample without a rolling path: a worked month, the exposure against the
# budget at FO_CURVE_BUDGETS (the three budgets and points between), and the
# feasibility of each budget in every month, from the forecast tracking error
# of the tracking-error-minimising solution under the mandate.
FO_CURVE_BUDGETS_ANNUAL = (0.0050, 0.0075, 0.0100, 0.0125, 0.0150, 0.0200)
FO_TOL_TE = 1e-9                         # a forecast tracking error within this of the budget counts as on it, in the tests
# Solver rule for the factor problem, added 18 October 2026 while notebook 15 was
# being built, before its first recorded run: the first path at 150 basis points
# stopped at 1981-02, a month in which the budget cone and the turnover cap are
# both active and the interior-point method stalls at a primal residual of 1e-7
# (status AlmostSolved at the 1e-9 rule, and a failure outright on the second pass
# at the default rule). The factor problem therefore gets a third pass at this
# looser rule when the solver fails at the first two; a portfolio from any pass is
# accepted only after the same checks as every other (the bounds, the cap, the
# budget within 1e-6 relative), and the month keeps the solver's status word, so
# that the notebook counts it among the inaccurate months. A second case met the
# same day, at 2009-06 for the principal-components estimator at 100 basis
# points: the minimiser's forecast exceeds the budget by a hundredth of a basis
# point, the factor problem has no interior, and the solver reports a numerical
# error at every rule; the month then follows the infeasibility rule above, as
# any month whose minimiser exceeds the budget does, and the guard that the
# minimiser must indeed exceed the budget stays, so that a solver failure on a
# feasible month still stops the run.
FO_SOLVER_OPTIONS_LOOSE = {"tol_gap_abs": 1e-7, "tol_gap_rel": 1e-7, "tol_feas": 1e-7}
# Notebook 15 (built 18 October 2026, after notebook 14 and the restatement of
# 9 October, before any path ran) runs the 19 paths month by month as notebook
# 12 runs version 1's, with the betas of the month before each portfolio, and
# judges E1, E2, E3, E5, E6 and E7 on their records; E4 needs the attribution
# of notebook 16 and is judged there. The gap of FO_GAP_DEFINITION is recorded
# in notebook 15, where the next month's betas are at hand, and drawn in
# notebook 16. The incidental value of the diagnostic is the exposure the
# mandate's tracking-error-minimising path of the same variant carries in the
# same month, rebuilt from the modules as notebook 13 rebuilds it and checked
# against ATTRIBUTION_REFERENCE_TE; the months the mandate alone can meet, E1's
# denominator, are those in which its stationary minimum forecast tracking
# error (the mandate with turnover free, as notebook 14 computes it) is at or
# below the budget.
FO_NOTEBOOK_15_JUDGES = ("E1", "E2", "E3", "E5", "E6", "E7")
FO_NOTEBOOK_16_JUDGES = ("E4",)
# Notebook 16, the attribution of the nineteen paths, built 8 November 2026,
# after the recorded run of notebook 15 and the design's reading of it on 6
# November, before any attribution ran. It rebuilds the nineteen paths from the
# modules, as notebook 13 rebuilds version 1's, checks each path's realised
# tracking error against notebook 15's below within ATTRIBUTION_REPRODUCTION_TOL
# and, where the repository is present, the records against the committed
# monthly table under the rule for reruns; it attributes each path's active
# return exactly as notebook 13 does and judges E4. Two measures it reports,
# in no claim: the exposure bought per factor, the mean active exposure to
# each targeted factor divided by the realised tracking error in basis points
# a year, the per-factor form of notebook 15's exposure per basis point; and
# the gap per unit of exposure bought, the path's mean absolute gap divided by
# its mean factor objective, which the design of 6 November 2026 allows as a
# reported measure with no expectation attached.
FO_ATTRIBUTION_DATE = "2026-11-08"
FO_ATTRIBUTION_REFERENCE_TE = {                  # notebook 15's realised tracking errors, per year, for the reproduction check
    "sample, daily 3y | 75": 0.01002117, "sample, daily 3y | 100": 0.01205713, "sample, daily 3y | 150": 0.01731365,
    "ledoit_wolf, daily 3y | 75": 0.00985564, "ledoit_wolf, daily 3y | 100": 0.01185552, "ledoit_wolf, daily 3y | 150": 0.01713197,
    "factor, daily 3y | 75": 0.01102305, "factor, daily 3y | 100": 0.01335962, "factor, daily 3y | 150": 0.01876902,
    "pca, daily 3y | 75": 0.01126859, "pca, daily 3y | 100": 0.01369633, "pca, daily 3y | 150": 0.01849629,
    "sample, monthly 120 | 75": 0.01279599, "sample, monthly 120 | 100": 0.01501193, "sample, monthly 120 | 150": 0.01917291,
    "robust, all five | 100": 0.01156417, "RiskMetrics, daily 5y | 100": 0.01186274,
    "sample, daily 3y, value alone | 100": 0.01227800, "sample, daily 3y, momentum alone | 100": 0.01230211,
}
FO_EXPOSURE_BOUGHT_PER_FACTOR = "mean active exposure to the factor over the realised tracking error in basis points a year"
FO_GAP_PER_UNIT = "mean absolute gap over the mean factor objective"
# Notebook 12, added 8 November 2026 in no claim, for the dashboard's figure of
# the forecasts' miss by decade in one measure for both of its states: the bias
# statistic by decade under the 21 rule and after calibration, for every path under C3 and the tilt alone,
# written as a table beside the calibration table, which stays as committed.
EVAL_BIAS_BY_DECADE_DATE = "2026-11-08"

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = cut_to_sample_end(chosen["frame"].copy())
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def cut_to_sample_end(frame: pd.DataFrame, end: str | None = None) -> pd.DataFrame:
    """The frame up to the end of the sample (constants.EXT_SAMPLE_END, fixed 6 November 2026): a monthly block to that
    month, a daily block to the last day of that month, an annual block to that year. A later vintage of French's
    files carries months after the end, which the study does not use; the vintage in the provenance record still says
    which revision of history the months before the end came from. With no end set, the frame is returned as it is."""
    end = C.EXT_SAMPLE_END if end is None else end
    if end is None or not isinstance(frame.index, pd.PeriodIndex) or len(frame) == 0:
        return frame
    end_m = pd.Period(end, "M")
    freq = frame.index.freqstr or ""
    if freq.startswith("M"):
        limit = end_m
    elif freq.startswith("D"):
        limit = end_m.asfreq("D", "end")
    else:
        limit = end_m.asfreq(frame.index.freq, "end")
    return frame.loc[:limit]


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/benchmark.py
"""The benchmark of the extension: the cap-weighted combination of French's 49 industry portfolios.

Each industry portfolio's value-weighted return for a month is the return of the
firms in the industry, each counted in proportion to its market equity at the
start of the month. Weighting the 49 industry returns by each industry's total
market equity at the same time (firm count times average firm size, two blocks of
the same French file) gives the value-weighted return of all the firms in the 49
portfolios, which should be close to French's market return. Notebook 08 measures
how close, against the level fixed in constants.CAPW_TE_MAX_ANNUAL, and decides
whether these weights are the benchmark of notebooks 09 to 13.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import french_loader as fl

PERIODS_PER_YEAR = 12


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame | pd.Series]:
    """The three monthly blocks of the 49-industry file and the market series.

    returns: value-weighted industry returns (decimal); firms: number of firms;
    size: average firm size in millions of dollars; market: Mkt-RF plus RF from
    the factors file (decimal). Missing codes are NaN in all of them.
    """
    returns = fl.load_monthly("ind49", r"Value Weighted Returns -- Monthly", cache_dir)
    firms = fl.load_counts("ind49", r"Number of Firms", cache_dir)
    size = fl.load_counts("ind49", r"Average Firm Size", cache_dir)
    f3 = fl.load_monthly("factors3", 0, cache_dir)
    excess, rf = C.CAPW_MARKET_SERIES
    market = (f3[excess] + f3[rf]).rename("market")
    return {"returns": returns, "firms": firms, "size": size, "market": market}


def market_equity(firms: pd.DataFrame, size: pd.DataFrame) -> pd.DataFrame:
    """Total market equity of each industry, in the units of the size block (millions of dollars)."""
    if not firms.index.equals(size.index) or not firms.columns.equals(size.columns):
        raise ValueError("firm counts and average sizes must share months and industries")
    return firms * size


def cap_weights(firms: pd.DataFrame, size: pd.DataFrame, timing: str = C.CAPW_WEIGHT_TIMING) -> pd.DataFrame:
    """Each industry's share of total market equity, per month; the rows sum to one.

    timing "same_month": the count and size of month t weight the returns of
    month t (French measures both at the start of the month). "previous_month":
    the count and size of month t-1 weight the returns of month t; the first
    month has no weights. Any month in which an industry's count or size is
    missing has no weights at all, so that a partial sum never passes as a
    whole-market weight.
    """
    me = market_equity(firms, size)
    if timing == "previous_month":
        me = me.shift(1)
    elif timing != "same_month":
        raise ValueError(f"unknown timing {timing!r}")
    total = me.sum(axis=1, min_count=me.shape[1])
    return me.div(total, axis=0)


def combination_return(returns: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    """Sum over industries of weight times return, per month; NaN where any of the 49 is missing."""
    if not returns.columns.equals(weights.columns):
        raise ValueError("returns and weights must have the same industries in the same order")
    aligned = weights.reindex(returns.index)
    return (aligned * returns).sum(axis=1, min_count=returns.shape[1]).rename("combination")


def tracking_error(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> float:
    """Annualised standard deviation of the per-period difference a - b, over the periods both have."""
    d = (a - b).dropna()
    if len(d) < 2:
        return float("nan")
    return float(d.std(ddof=1) * np.sqrt(periods_per_year))


def compare(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> dict[str, float]:
    """Tracking error, annualised mean difference, correlation, largest absolute monthly difference and month count of a against b."""
    both = pd.concat([a, b], axis=1).dropna()
    d = both.iloc[:, 0] - both.iloc[:, 1]
    return {
        "months": int(len(d)),
        "tracking_error_annual": float(d.std(ddof=1) * np.sqrt(periods_per_year)),
        "mean_difference_annual": float(d.mean() * periods_per_year),
        "correlation": float(both.iloc[:, 0].corr(both.iloc[:, 1])) if both.std(ddof=1).gt(0).all() else float("nan"),
        "max_abs_monthly_difference": float(d.abs().max()),
        "month_of_max": str(d.abs().idxmax()),
    }


def weight_extremes(weights: pd.DataFrame, month: str | pd.Period) -> dict[str, object]:
    """The largest and smallest weights in one month, with their industries."""
    row = weights.loc[pd.Period(month, "M")].dropna()
    return {
        "month": str(pd.Period(month, "M")),
        "largest": row.idxmax(), "largest_weight": float(row.max()),
        "smallest": row.idxmin(), "smallest_weight": float(row.min()),
        "industries": int(len(row)),
    }

In [ ]:
%%writefile src/bp/rules.py
"""
Portfolio rules from DeMiguel, Garlappi and Uppal (2009), section 1.

Each rule is a function of one estimation window of monthly excess returns,
an array of shape (M, N), and returns the vector of relative weights on the N
risky assets, DGU equation (1): the mean-variance solution x = Sigma^-1 mu / gamma
normalised by the absolute value of its sum, so that the weights sum to +1 or
to -1 and the direction of the position is preserved in the few windows where
the risky assets are net short.

Notation follows the paper: mu and Sigma are the sample moments of the window,
M its length, N the number of assets. The risk aversion gamma cancels in the
normalisation for every unconstrained rule, which is why it does not appear.

Rules in this module, with the DGU abbreviation and section:
  ew       1/N, section 1.1
  mv       sample-based mean-variance, section 1.2
  bs       Bayes-Stein shrinkage, section 1.3.2, Jorion (1986)
  min      minimum variance, section 1.4.1
  vw       the value-weighted market, section 1.4.2 (all weight on the market column)
  mv-c     short-sale-constrained mean-variance, section 1.5
  bs-c     short-sale-constrained Bayes-Stein, section 1.5
  min-c    short-sale-constrained minimum variance, section 1.5
  g-min-c  minimum variance with every weight at least 1/(2N), section 1.5

The constrained rules are quadratic programmes. Each is solved exactly with
the Lawson-Hanson non-negative least squares algorithm (scipy.optimize.nnls)
after a change of variables, so there is no solver setting to tune:
  minimise  x' Sigma x / 2 - mu' x   subject to x >= 0
is, with Sigma = L L' (Cholesky), the same as minimising |L' x - L^-1 mu|^2,
a non-negative least squares problem. A budget constraint 1'w = 1 is added
as one heavily weighted extra row, then the result is renormalised; a lower
bound w >= a is a shift of variables. tests/ checks every one against an
independent solver.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import nnls

from . import constants as C


def sample_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sample mean and covariance (divisor M - 1) of an (M, N) window."""
    R = np.asarray(R, dtype=float)
    mu = R.mean(axis=0)
    Sigma = np.cov(R, rowvar=False, ddof=1)
    return mu, np.atleast_2d(Sigma)


def normalise(x: np.ndarray) -> np.ndarray:
    """DGU equation (1): w = x / |1'x|."""
    s = float(np.sum(x))
    if s == 0.0:
        raise ZeroDivisionError("portfolio weights sum to zero; cannot normalise")
    return x / abs(s)


def ew(R: np.ndarray) -> np.ndarray:
    n = np.asarray(R).shape[1]
    return np.full(n, 1.0 / n)


def mv(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    x = np.linalg.solve(Sigma, mu)
    return normalise(x)


def min_variance(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    ones = np.ones(Sigma.shape[0])
    x = np.linalg.solve(Sigma, ones)
    return x / x.sum()


def vw(market_index: int):
    """The value-weighted market: weight one on the market column, zero elsewhere."""
    def rule(R: np.ndarray) -> np.ndarray:
        w = np.zeros(np.asarray(R).shape[1])
        w[market_index] = 1.0
        return w
    rule.__name__ = "vw"
    return rule


def in_sample_sharpe(R: np.ndarray, rule=mv) -> float:
    """DGU equation (13): the Sharpe ratio of a rule estimated once on the whole sample."""
    mu, Sigma = sample_moments(R)
    w = rule(R)
    return float(w @ mu / np.sqrt(w @ Sigma @ w))


# ---------------------------------------------------------------------------
# Bayes-Stein, DGU section 1.3.2, equations (4) and (5); Jorion (1986)
# ---------------------------------------------------------------------------

def bayes_stein_from_moments(mu: np.ndarray, Sigma_mnm2: np.ndarray, M: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Jorion's predictive moments from a window's mean and its covariance with divisor M - N - 2.

    Returns (mu_bs, Sigma_bs, phi). The mean is shrunk toward the mean return
    of the minimum-variance portfolio by phi (DGU equation 5); the covariance
    is inflated for the uncertainty in the mean, Jorion's predictive variance.
    """
    N = len(mu)
    Sigma_inv_1 = np.linalg.solve(Sigma_mnm2, np.ones(N))
    w_min = Sigma_inv_1 / Sigma_inv_1.sum()
    mu_min = float(mu @ w_min)
    d = mu - mu_min
    dist = float(d @ np.linalg.solve(Sigma_mnm2, d))     # (mu - mu_min 1)' Sigma^-1 (mu - mu_min 1)
    lam = (N + 2) / dist
    phi = lam / (M + lam)                                 # equals (N+2) / ((N+2) + M * dist), DGU equation (5)
    mu_bs = (1.0 - phi) * mu + phi * mu_min
    Sigma_bs = Sigma_mnm2 * (1.0 + 1.0 / (M + lam)) + (lam / (M * (M + 1.0 + lam))) * np.outer(np.ones(N), np.ones(N)) / Sigma_inv_1.sum()
    return mu_bs, Sigma_bs, phi


def bayes_stein_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """The predictive mean and covariance of Jorion (1986), as DGU implement them, from a window of returns.

    The sample covariance here divides by M - N - 2 (DGU equation 5), the
    unbiased estimator of the inverse. Returns (mu_bs, Sigma_bs, phi).
    """
    R = np.asarray(R, dtype=float)
    M, N = R.shape
    mu = R.mean(axis=0)
    dev = R - mu
    Sigma = dev.T @ dev / (M - N - 2)
    return bayes_stein_from_moments(mu, Sigma, M)


def bs(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


# ---------------------------------------------------------------------------
# Constrained rules, DGU section 1.5, solved as non-negative least squares
# ---------------------------------------------------------------------------

_BUDGET_WEIGHT = 1e6   # weight of the budget row; the residual on 1'w = 1 is then below 1e-12 and the result is renormalised anyway


def _cholesky_upper(Sigma: np.ndarray) -> np.ndarray:
    """L' with Sigma = L L'. A tiny ridge keeps a near-singular window solvable."""
    N = Sigma.shape[0]
    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(Sigma + 1e-12 * np.trace(Sigma) / N * np.eye(N))
    return L.T


def nonneg_mean_variance(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """argmax mu'x - x'Sigma x / 2 over x >= 0, as NNLS: minimise |L'x - L^-1 mu|^2.

    This is DGU's Lagrangian (8) read literally: the non-negativity constraint
    alone, the scale then removed by normalise(). It is kept so that the two
    readings of the equation can be compared (notebook 03); budget_mean_variance
    is the reading that reproduces the paper's tables.
    """
    Lt = _cholesky_upper(Sigma)
    b = np.linalg.solve(Lt.T, mu)           # L^-1 mu
    x, _ = nnls(Lt, b)
    return x


def budget_mean_variance(mu: np.ndarray, Sigma: np.ndarray, gamma: float = C.DGU_RISK_AVERSION) -> np.ndarray:
    """argmax mu'w - gamma w'Sigma w / 2 over w >= 0 and 1'w = 1.

    The budget constraint sits inside the optimisation, so gamma no longer
    cancels: with gamma = 1 and monthly moments the risk term is small against
    the differences in means, and the rule often puts all wealth in the asset
    with the highest estimated mean. That is the "corner solutions with all
    wealth invested in a single asset" of DGU footnote 22, and it is what
    reproduces their Tables 3 to 5 (constants.DGU_CONSTRAINED_BUDGET_INSIDE).
    As NNLS: minimise |sqrt(gamma) L'w - L^-1 mu / sqrt(gamma)|^2 with the
    budget as a heavily weighted extra row.
    """
    N = len(mu)
    Lt = _cholesky_upper(Sigma)
    A = np.vstack([np.sqrt(gamma) * Lt, _BUDGET_WEIGHT * np.ones(N)])
    b = np.concatenate([np.linalg.solve(Lt.T, mu) / np.sqrt(gamma), [_BUDGET_WEIGHT]])
    w, _ = nnls(A, b)
    return w / w.sum()


def constrained_min_variance(Sigma: np.ndarray, lower: float = 0.0) -> np.ndarray:
    """argmin w'Sigma w over w >= lower, 1'w = 1, as NNLS on v = w - lower.

    Objective |L'(v + lower 1)|^2, so the NNLS target is -L' lower 1; the budget
    1'v = 1 - N lower is appended as a heavily weighted row.
    """
    N = Sigma.shape[0]
    Lt = _cholesky_upper(Sigma)
    ones = np.ones(N)
    A = np.vstack([Lt, _BUDGET_WEIGHT * ones])
    b = np.concatenate([-Lt @ (lower * ones), [_BUDGET_WEIGHT * (1.0 - N * lower)]])
    v, _ = nnls(A, b)
    w = v + lower
    return w / w.sum()


def mv_c(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def mv_c_cone(R: np.ndarray) -> np.ndarray:
    """Equation (8) read literally, for the comparison in notebook 03; not a reported rule."""
    mu, Sigma = sample_moments(R)
    return normalise(nonneg_mean_variance(mu, Sigma))


def min_c(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c(R: np.ndarray) -> np.ndarray:
    """DGU p. 1926: minimum variance with w >= a 1, a = 1/(2N), halfway between min-c and 1/N."""
    _, Sigma = sample_moments(R)
    N = Sigma.shape[0]
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / N)


# ---------------------------------------------------------------------------
# The same rules from a window's moments, for the long simulated histories of
# notebook 04, where recomputing the covariance at every month would dominate
# the running time. Each takes (mu, Sigma, M) with Sigma the sample covariance
# with divisor M - 1, and returns the same weights as its window-based twin
# (tests/test_rules_backtest.py checks the equality).
# ---------------------------------------------------------------------------

def _to_mnm2(Sigma: np.ndarray, M: int) -> np.ndarray:
    """Rescale a divisor-(M-1) covariance to divisor M - N - 2, DGU equation (5)."""
    N = Sigma.shape[0]
    return Sigma * (M - 1) / (M - N - 2)


def ew_m(mu, Sigma, M):
    return np.full(len(mu), 1.0 / len(mu))


def mv_m(mu, Sigma, M):
    return normalise(np.linalg.solve(Sigma, mu))


def min_m(mu, Sigma, M):
    x = np.linalg.solve(Sigma, np.ones(len(mu)))
    return x / x.sum()


def bs_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


def mv_c_m(mu, Sigma, M):
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / len(mu))


MOMENT_RULES = {
    "ew": ew_m,
    "mv": mv_m,
    "bs": bs_m,
    "min": min_m,
    "mv-c": mv_c_m,
    "bs-c": bs_c_m,
    "min-c": min_c_m,
    "g-min-c": g_min_c_m,
}


# The registry. vw needs the market column and is built per dataset with vw(index).
RULES = {
    "ew": ew,
    "mv": mv,
    "bs": bs,
    "min": min_variance,
    "mv-c": mv_c,
    "bs-c": bs_c,
    "min-c": min_c,
    "g-min-c": g_min_c,
}

In [ ]:
%%writefile src/bp/covariance.py
"""Four ways to estimate a covariance matrix from a window of excess returns, and the tests that judge them (notebook 10).

Every estimator takes a window of excess returns (rows are periods, columns are
assets) and returns an N by N covariance matrix for the period that follows. The
sample covariance uses all N(N+1)/2 numbers the window offers. Ledoit-Wolf
shrinkage pulls it towards a simple target. The factor model and the
principal-component model describe it with a small number of common sources of movement
plus each asset's own variance. The tests measure how well each estimate
forecasts the variance of test portfolios in the next month (the bias
statistic) and how well the minimum-variance portfolio built from it does
(the tests of Dom, Howard, Jansen and Lohre, 2024).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import rules as S

PERIODS_PER_YEAR = 12


# ---------------------------------------------------------------------------
# Estimators
# ---------------------------------------------------------------------------

def sample_cov(X: np.ndarray) -> np.ndarray:
    """The sample covariance, denominator T - 1."""
    return np.cov(np.asarray(X, dtype=float), rowvar=False, ddof=1)


def ewma_cov(X: np.ndarray, halflife: float) -> np.ndarray:
    """Exponentially weighted covariance: the weight of an observation halves every `halflife` periods, the most recent weighing most.

    The mean is weighted the same way; weights are normalised to sum to one.
    """
    X = np.asarray(X, dtype=float)
    T = X.shape[0]
    lam = 0.5 ** (1.0 / halflife)
    w = lam ** np.arange(T - 1, -1, -1)
    w = w / w.sum()
    mu = w @ X
    D = X - mu
    return (D * w[:, None]).T @ D / (1.0 - (w ** 2).sum())   # the unbiased normalisation for weighted samples


def lw_scaled_identity(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004b): (1 - k) S + k m I, with S the sample covariance (denominator T), m the average
    sample variance and k the intensity that minimises the expected squared error.

    Returns the estimate and k. The intensity is b^2 / d^2 clipped to [0, 1], where d^2 is the squared
    distance between S and m I and b^2 measures how much S moves from one observation to the next.
    """
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    m = np.trace(S) / N
    d2 = ((S - m * np.eye(N)) ** 2).sum() / N
    b2_bar = sum(((np.outer(x, x) - S) ** 2).sum() for x in D) / (N * T * T)
    b2 = min(b2_bar, d2)
    k = float(b2 / d2) if d2 > 0 else 0.0
    return (1.0 - k) * S + k * m * np.eye(N), k


def lw_constant_correlation(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004a): shrink S towards the matrix with the same variances and one common correlation,
    the average sample correlation; the intensity follows the paper's appendix. Returns the estimate and k."""
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    var = np.diag(S)
    sd = np.sqrt(var)
    corr = S / np.outer(sd, sd)
    rbar = (corr.sum() - N) / (N * (N - 1))
    F = rbar * np.outer(sd, sd)
    np.fill_diagonal(F, var)
    # pi: the sum over i, j of the estimated asymptotic variances of the entries of sqrt(T) S
    Y = D ** 2
    pi_mat = (Y.T @ Y) / T - S ** 2
    pi = pi_mat.sum()
    # rho: the diagonal part plus the covariance between the entries of S and the target's correlation structure
    term1 = ((D ** 3).T @ D) / T          # theta_{ii,ij}: average over t of x_it^3 x_jt
    help_ = term1 - var[:, None] * S      # (1/T) sum_t (x_it^2 - S_ii)(x_it x_jt - S_ij)
    theta_ii = help_
    theta_jj = help_.T
    rho_off = (rbar / 2.0) * (np.sqrt(var[None, :] / var[:, None]) * theta_ii + np.sqrt(var[:, None] / var[None, :]) * theta_jj)
    np.fill_diagonal(rho_off, 0.0)
    rho = np.trace(pi_mat) + rho_off.sum()
    gamma = ((F - S) ** 2).sum()
    k = (pi - rho) / gamma / T if gamma > 0 else 0.0
    k = float(min(1.0, max(0.0, k)))
    return (1.0 - k) * S + k * F, k


def factor_cov(X: np.ndarray, F: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """The exact factor model: X regressed on the factors F (with an intercept) over the window;
    the estimate is B Sigma_F B' plus the diagonal of residual variances. Returns the estimate and B (N by K)."""
    X = np.asarray(X, dtype=float)
    F = np.asarray(F, dtype=float)
    T, K = F.shape
    A = np.column_stack([np.ones(T), F])
    coef, *_ = np.linalg.lstsq(A, X, rcond=None)      # (K + 1) by N
    B = coef[1:].T                                     # N by K
    resid = X - A @ coef
    Sigma_F = np.cov(F, rowvar=False, ddof=1).reshape(K, K)
    resid_var = resid.var(axis=0, ddof=K + 1)
    return B @ Sigma_F @ B.T + np.diag(resid_var), B


def pca_cov(X: np.ndarray, n_components: int, market: np.ndarray | None = None) -> tuple[np.ndarray, int]:
    """The principal-component model.

    With `market` given (constants.PCA_FIRST_COMPONENT == "market"): each asset is regressed on the market
    series, and the covariance of the residuals is described by its first n_components - 1 principal
    components plus the diagonal of what they leave; the estimate is beta beta' var(market) plus that.
    Without a market series, the first n_components of X's own covariance plus the diagonal remainder.
    Returns the estimate and the number of components taken from the data.
    """
    X = np.asarray(X, dtype=float)
    if market is not None:
        m = np.asarray(market, dtype=float)
        A = np.column_stack([np.ones(len(m)), m])
        coef, *_ = np.linalg.lstsq(A, X, rcond=None)
        beta = coef[1]
        resid = X - A @ coef
        base = np.outer(beta, beta) * m.var(ddof=1)
        k = n_components - 1
        target = np.cov(resid, rowvar=False, ddof=2)
    else:
        base = 0.0
        k = n_components
        target = sample_cov(X)
    if k <= 0:
        return base + np.diag(np.diag(target)), 0
    vals, vecs = np.linalg.eigh(target)
    order = np.argsort(vals)[::-1][:k]
    V = vecs[:, order]
    L = np.clip(vals[order], 0.0, None)
    common = (V * L) @ V.T
    own = np.diag(np.clip(np.diag(target) - np.diag(common), 1e-12, None))
    return base + common + own, k


def scale_daily_to_monthly(Sigma_daily: np.ndarray, scale: float = C.DAILY_TO_MONTHLY_SCALE) -> np.ndarray:
    """A daily covariance times the number of trading days in a month; exact when daily returns do not predict the next day's."""
    return Sigma_daily * scale


def variance_ratio(x: np.ndarray, q: int = C.COV_SCALE_VR_HORIZON_DAYS) -> float:
    """The variance ratio of Lo and MacKinlay (1988): the variance of the overlapping sums of q consecutive
    values divided by q times the variance of the single values. It is 1 when the values do not predict each
    other, above 1 when a high value tends to be followed by a high one (positive autocorrelation) and below 1
    when a high value tends to be followed by a low one. Multiplying a daily covariance by q times this ratio,
    in place of q alone, gives the covariance of q-day sums that the autocorrelation implies."""
    v = np.asarray(x, dtype=float)
    if len(v) < 2 * q:
        return float("nan")
    sums = np.convolve(v, np.ones(q), mode="valid")
    return float(sums.var(ddof=1) / (q * v.var(ddof=1)))


# ---------------------------------------------------------------------------
# Windows
# ---------------------------------------------------------------------------

def monthly_window(excess_m: pd.DataFrame, month: pd.Period, months: int = C.EXT_ESTIMATION_WINDOW) -> pd.DataFrame:
    """The `months` monthly rows before `month`."""
    end = month - 1
    start = month - months
    return excess_m.loc[start:end]


def daily_window(excess_d: pd.DataFrame, month: pd.Period, years: int) -> pd.DataFrame:
    """The trading days of the 12 * years calendar months before `month`, days with any missing value left out."""
    end = (month - 1).asfreq("D", "end")
    start = (month - 12 * years).asfreq("D", "start")
    return excess_d.loc[start:end].dropna()


def daily_benchmark_excess(returns_d: pd.DataFrame, rf_d: pd.Series, weights_m: pd.DataFrame) -> pd.Series:
    """The benchmark's daily excess return: each day's industry returns weighted by that month's benchmark weights, minus the daily risk-free rate."""
    month_of_day = returns_d.index.asfreq("M")
    w = weights_m.reindex(month_of_day)
    w.index = returns_d.index
    return ((w * returns_d).sum(axis=1, min_count=returns_d.shape[1]) - rf_d.reindex(returns_d.index)).rename("benchmark_excess")


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def bias_statistic(realised: np.ndarray, predicted_var: np.ndarray) -> float:
    """The standard deviation of realised / sqrt(predicted variance): 1 when the forecasts are calibrated, above 1 when risk is under-forecast."""
    z = np.asarray(realised, dtype=float) / np.sqrt(np.asarray(predicted_var, dtype=float))
    return float(z.std(ddof=1))


def random_active_weights(n_assets: int, draws: int = C.COV_TEST_ACTIVE_DRAWS, bound: float = C.ACTIVE_WEIGHT_BOUND,
                          seed: int = C.COV_TEST_SEED) -> np.ndarray:
    """`draws` active-weight vectors: weights sum to zero and the largest absolute weight equals `bound`."""
    rng = np.random.default_rng(seed)
    U = rng.uniform(-1.0, 1.0, size=(draws, n_assets))
    U = U - U.mean(axis=1, keepdims=True)
    return bound * U / np.abs(U).max(axis=1, keepdims=True)


def min_variance_unconstrained(Sigma: np.ndarray) -> np.ndarray:
    """w proportional to Sigma^-1 1, scaled to sum to one; negative weights allowed."""
    x = np.linalg.solve(Sigma, np.ones(Sigma.shape[0]))
    return x / x.sum()


def min_variance_long_only(Sigma: np.ndarray) -> np.ndarray:
    """The long-only minimum-variance portfolio, solved exactly by non-negative least squares (rules.constrained_min_variance)."""
    return S.constrained_min_variance(Sigma, 0.0)


def condition_number(Sigma: np.ndarray) -> float:
    vals = np.linalg.eigvalsh(Sigma)
    return float(vals.max() / max(vals.min(), 1e-300))


def annualised_vol(x: pd.Series | np.ndarray) -> float:
    return float(np.std(np.asarray(x, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


# ---------------------------------------------------------------------------
# The panels and the estimator variants, as notebooks 10 to 12 use them
# ---------------------------------------------------------------------------

class Panels:
    """Everything the estimators read: monthly and daily excess returns of the 49 industries, the six factors at both
    frequencies, the benchmark's weights and its excess return at both frequencies, and the plain monthly returns."""

    def __init__(self, start: str = C.EXT_SAMPLE_START):
        from . import french_loader as fl
        from . import benchmark as BM
        inputs = BM.load_inputs()
        f3 = fl.load_monthly("factors3", 0)
        f5 = fl.load_monthly("factors5", 0)
        mom = fl.load_monthly("momentum", 0)
        start_p = pd.Period(start, "M")
        self.returns_m = inputs["returns"].loc[start_p:]
        self.last = self.returns_m.index[-1]
        self.rf_m = f3["RF"].loc[start_p:self.last]
        self.excess_m = self.returns_m.sub(self.rf_m, axis=0)
        self.weights = BM.cap_weights(inputs["firms"].loc[start_p:], inputs["size"].loc[start_p:], C.CAPW_WEIGHT_TIMING)
        self.bench_m = (BM.combination_return(self.returns_m, self.weights) - self.rf_m).rename("benchmark_excess")
        self.factors_m = pd.concat([f5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1).loc[start_p:self.last]
        returns_d = fl.load_monthly("ind49_daily", r"Value Weighted Returns -- Daily").loc[start_p.asfreq("D", "start"):]
        f3d = fl.load_monthly("factors3_daily", 0)
        f5d = fl.load_monthly("factors5_daily", 0)
        momd = fl.load_monthly("momentum_daily", 0)
        self.rf_d = f3d["RF"].reindex(returns_d.index)
        self.returns_d = returns_d
        self.excess_d = returns_d.sub(self.rf_d, axis=0)
        self.factors_d = pd.concat([f5d[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], momd["Mom"]], axis=1).reindex(returns_d.index)
        self.bench_d = daily_benchmark_excess(returns_d, self.rf_d, self.weights)
        self.files = ("ind49", "factors3", "factors5", "momentum", "ind49_daily", "factors3_daily", "factors5_daily", "momentum_daily")


DAILY_YEARS = {"daily_3y": C.EXT_COV_DAILY_WINDOW_YEARS, "daily_1y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[0],
               "daily_5y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1], "daily_5y_ewma": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1]}


def estimate(name: str, freq: str, month: pd.Period, P: Panels) -> tuple[np.ndarray, float, int]:
    """One estimator variant for `month`, built on the data before it: the monthly covariance (daily ones scaled by
    DAILY_TO_MONTHLY_SCALE), the shrinkage intensity (nan for the others) and the number of observations in the window.
    The same dispatch as notebook 10's evaluation loop."""
    if freq == "monthly_120":
        X = monthly_window(P.excess_m, month); F = P.factors_m.loc[X.index]; mk = P.bench_m.loc[X.index]; scale = 1.0
    else:
        X = daily_window(P.excess_d, month, DAILY_YEARS[freq]); F = P.factors_d.loc[X.index]; mk = P.bench_d.loc[X.index]; scale = C.DAILY_TO_MONTHLY_SCALE
    Xv = X.to_numpy(); intensity = np.nan
    if name == "sample":
        S = ewma_cov(Xv, C.EXT_COV_EWMA_HALFLIFE_DAYS) if freq.endswith("ewma") else sample_cov(Xv)
    elif name == "ledoit_wolf":
        S, intensity = lw_scaled_identity(Xv)
    elif name == "ledoit_wolf_cc":
        S, intensity = lw_constant_correlation(Xv)
    elif name == "factor":
        S, _ = factor_cov(Xv, F.to_numpy())
    elif name == "pca":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS, market=mk.to_numpy())
    elif name == "pca_2":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS_SENSITIVITY, market=mk.to_numpy())
    else:
        raise KeyError(name)
    return S * scale, intensity, len(X)


# The five estimator variants of the rolling evaluation (notebook 12): the four estimators on three years of daily
# returns and the paper's convention, the sample covariance of 120 monthly returns.
EVALUATION_VARIANTS = tuple((e, "daily_3y") for e in C.COV_ESTIMATORS) + (("sample", "monthly_120"),)
# The robust portfolio (constants.ROBUST_VARIANT) is built from all five covariances at once, and RiskMetrics
# (constants.EWMA_VARIANT) is the recency-weighted sample covariance; the seven together are the variants the rolling evaluation reports.
ALL_VARIANTS = EVALUATION_VARIANTS + (C.ROBUST_VARIANT, C.EWMA_VARIANT)

In [ ]:
%%writefile src/bp/optimiser.py
"""The optimiser of the extension: the portfolio that tracks the benchmark most closely under a set of constraints (notebook 11).

For a month with benchmark weights b and a monthly covariance matrix Sigma, the
optimiser finds the weights w that minimise the forecast tracking error, the
square root of 12 (w - b)' Sigma (w - b), subject to the constraints of a named
set: the weights sum to one; no short positions (long_only); each untilted
industry within ACTIVE_WEIGHT_BOUND of its benchmark weight (active_weight_bound);
one-way turnover from the drifted previous portfolio at most
TURNOVER_CAP_MONTHLY_ONE_WAY (turnover_cap); the tilted industries at no more than
TILT_MAX_SHARE_OF_BENCHMARK of their benchmark weight (tilt); named industries
at zero (exclusion); the portfolio's beta to the benchmark equal to the benchmark's
(beta_neutral). The objective is quadratic and every constraint is linear,
so the problem is a convex quadratic program with one global minimum, solved by
cvxpy with the solver named in constants.OPT_SOLVER. Two safeguards (notebook 11,
check D5): when the turnover cap is among the constraints, the smallest feasible
one-way turnover is found first, by a linear program, so that the solver is never
handed a capped problem that has no solution (a cap below that turnover is raised
to it and the month is marked relaxed); and every solution is checked against its
constraints within constants.OPT_TOL before it is returned, whatever status the
solver reports.
"""
from __future__ import annotations

import warnings
from dataclasses import dataclass, field

import cvxpy as cp
import numpy as np

from . import constants as C

PERIODS_PER_YEAR = 12


class OptimiserError(RuntimeError):
    """The solver did not return an optimal solution, or the solution failed a check."""


@dataclass
class Solution:
    weights: np.ndarray
    active: np.ndarray                 # weights minus benchmark weights
    tracking_error: float              # forecast, per year: sqrt(12 a' Sigma a)
    turnover: float                    # one-way, against the previous portfolio (nan when there is none)
    turnover_cap_used: float | None    # the cap applied, after any relaxation
    relaxed: bool                      # True when the cap had to be raised to make the month feasible
    status: str
    binding: dict = field(default_factory=dict)   # constraint name -> count of bounds within OPT_TOL of their limit


def cholesky_factor(Sigma: np.ndarray) -> np.ndarray:
    """L with Sigma = L L', after symmetrising; a jitter of 1e-10 on the diagonal is added only if Sigma is not positive definite."""
    S = 0.5 * (Sigma + Sigma.T)
    try:
        return np.linalg.cholesky(S)
    except np.linalg.LinAlgError:
        return np.linalg.cholesky(S + 1e-10 * np.eye(S.shape[0]))


def drifted_weights(w_prev: np.ndarray, returns_prev: np.ndarray) -> np.ndarray:
    """The previous portfolio after the month's returns: w_i (1 + r_i) divided by the portfolio's own gross return."""
    grown = np.asarray(w_prev, dtype=float) * (1.0 + np.asarray(returns_prev, dtype=float))
    return grown / grown.sum()


def forecast_tracking_error(w: np.ndarray, b: np.ndarray, Sigma: np.ndarray) -> float:
    """sqrt(12 a' Sigma a) with a = w - b, in return units per year."""
    a = np.asarray(w, dtype=float) - np.asarray(b, dtype=float)
    return float(np.sqrt(max(PERIODS_PER_YEAR * a @ Sigma @ a, 0.0)))


def effective_number(w: np.ndarray) -> float:
    """1 / sum of squared weights: 49 for 1/N over 49 industries, smaller for a concentrated portfolio."""
    w = np.asarray(w, dtype=float)
    return float(1.0 / (w @ w))


def one_way_turnover(w: np.ndarray, w0: np.ndarray) -> float:
    """Half the sum of absolute weight changes: the fraction of the portfolio sold (and bought) at the rebalance."""
    return float(0.5 * np.abs(np.asarray(w, dtype=float) - np.asarray(w0, dtype=float)).sum())


def benchmark_betas(Sigma: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Each asset's beta to the benchmark under Sigma: Sigma b divided by b' Sigma b; the benchmark's own beta is one."""
    Sb = np.asarray(Sigma, dtype=float) @ np.asarray(b, dtype=float)
    return Sb / float(np.asarray(b, dtype=float) @ Sb)


def active_variance_parts(a: np.ndarray, Sigma: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    """Split a' Sigma a into the market part, (a' beta)^2 times the benchmark's variance, and the rest.

    With beta = Sigma b / (b' Sigma b) the two parts add up to a' Sigma a exactly: the market part is the
    variance of the active return's projection on the benchmark, the rest is the variance of what the
    benchmark leaves unexplained."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float); Sigma = np.asarray(Sigma, dtype=float)
    beta = benchmark_betas(Sigma, b)
    var_b = float(b @ Sigma @ b)
    market = float((a @ beta) ** 2 * var_b)
    total = float(a @ Sigma @ a)
    return market, total - market


def bounds(b: np.ndarray, constraints: tuple, tilt_mask: np.ndarray, exclude_mask: np.ndarray,
           active_bound: float = C.ACTIVE_WEIGHT_BOUND, tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK) -> tuple[np.ndarray, np.ndarray]:
    """The lower and upper bound on each weight that the named constraints imply."""
    N = len(b)
    lower = np.full(N, -np.inf)
    upper = np.full(N, np.inf)
    if "long_only" in constraints:
        lower = np.maximum(lower, 0.0)
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        lower = np.where(untilted, np.maximum(lower, b - active_bound), lower)
        upper = np.where(untilted, np.minimum(upper, b + active_bound), upper)
    if "tilt" in constraints:
        lower = np.where(tilt_mask, np.maximum(lower, 0.0), lower)
        upper = np.where(tilt_mask, np.minimum(upper, tilt_share * b), upper)
    if "exclusion" in constraints:
        lower = np.where(exclude_mask, 0.0, lower)
        upper = np.where(exclude_mask, 0.0, upper)
    return lower, upper


def _build(Ls: list, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None,
           objective: str = "tracking_error", betas: list | None = None):
    """The cvxpy problem. `Ls` holds one Cholesky factor for the single-covariance problem and one per covariance for the robust
    one, whose objective is the largest of the forecast active variances; `betas` holds the beta vectors the
    beta-neutral constraint applies to, one per covariance."""
    N = len(b)
    w = cp.Variable(N)
    cons = [cp.sum(w) == 1]
    finite_lo = np.isfinite(lower)
    finite_up = np.isfinite(upper)
    if finite_lo.any():
        cons.append(w[finite_lo] >= lower[finite_lo])
    if finite_up.any():
        cons.append(w[finite_up] <= upper[finite_up])
    for beta in betas or []:
        cons.append(beta @ (w - b) == 0)
    turnover_expr = None
    if w0 is not None:
        # one-way turnover through explicit purchases p and sales n, w - w0 = p - n: the same quantity as half the sum of
        # absolute changes, in the form in which trades are written as buys and sells. The solver's status is not relied on
        # in either form (notebook 11, check D5: on a month with no feasible portfolio the solver reported "optimal" with
        # weights that fail the budget); check() tests every returned portfolio, and _solve poses a capped problem only
        # after the smallest feasible turnover is known.
        p = cp.Variable(N, nonneg=True)
        n = cp.Variable(N, nonneg=True)
        cons.append(w - w0 == p - n)
        turnover_expr = 0.5 * cp.sum(p + n)
        if cap is not None:
            cons.append(turnover_expr <= cap)
    if objective == "tracking_error" and len(Ls) == 1:
        obj = cp.Minimize(cp.sum_squares(Ls[0].T @ (w - b)))
    elif objective == "tracking_error":                     # robust: the largest active standard deviation across the covariances
        t = cp.Variable()                                   # written with norms (second-order cones) and in percent, which the solver handles more accurately than squares
        cons += [cp.norm(100.0 * (L.T @ (w - b)), 2) <= t for L in Ls]
        obj = cp.Minimize(t)
    else:                                                   # the smallest feasible one-way turnover
        obj = cp.Minimize(turnover_expr)
    return w, cp.Problem(obj, cons)


def solve(Sigma: np.ndarray, b: np.ndarray, constraints: tuple, w_prev: np.ndarray | None = None,
          tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
          turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
          tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> Solution:
    """The constrained tracking-error minimiser for one month.

    `constraints` is a tuple of names from constants.CONSTRAINT_SETS. `w_prev` is the previous portfolio after
    drift; it is required when "turnover_cap" is among the constraints. If the cap makes the month infeasible,
    the cap is raised to the smallest feasible one-way turnover and the solution says so.
    """
    return _solve([np.asarray(Sigma, dtype=float)], b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)


def solve_robust(Sigmas: list, b: np.ndarray, constraints: tuple, w_prev: np.ndarray | None = None,
                 tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
                 turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
                 tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> Solution:
    """The robust minimiser: the weights whose largest forecast active variance across the covariances in `Sigmas`
    is smallest, under the same constraints; beta neutrality, when asked, holds under the average of the covariances'
    beta vectors. The tracking error reported is the worst case across the covariances."""
    return _solve([np.asarray(S, dtype=float) for S in Sigmas], b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)


def _solve(Sigmas: list, b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol) -> Solution:
    b = np.asarray(b, dtype=float)
    N = len(b)
    tilt_mask = np.zeros(N, bool) if tilt_mask is None else np.asarray(tilt_mask, bool)
    exclude_mask = np.zeros(N, bool) if exclude_mask is None else np.asarray(exclude_mask, bool)
    if "turnover_cap" in constraints and w_prev is None:
        raise OptimiserError("the turnover cap needs the previous portfolio")
    if "exclusion" in constraints and (exclude_mask is None or not np.asarray(exclude_mask, bool).any()):
        raise OptimiserError("the exclusion constraint needs a mask naming at least one industry")
    lower, upper = bounds(b, constraints, tilt_mask, exclude_mask, active_bound, tilt_share)
    if (lower > upper + tol).any():
        raise OptimiserError("a weight's lower bound exceeds its upper bound; the constraint set is contradictory for this month")
    Ls = [cholesky_factor(S) for S in Sigmas]
    w0 = None if w_prev is None else np.asarray(w_prev, dtype=float)
    cap = turnover_cap if "turnover_cap" in constraints else None
    # beta neutrality: one equality. With one covariance, its own betas; with more than one (the robust problem), the average
    # of their beta vectors, because one equality per covariance made the 1979-07 problem infeasible under the mandate's
    # bounds, and the minimax objective already penalises the market exposure that remains under each covariance.
    betas = [np.mean([benchmark_betas(S, b) for S in Sigmas], axis=0)] if "beta_neutral" in constraints else None
    relaxed = False

    options = C.OPT_SOLVER_OPTIONS if len(Ls) == 1 else C.OPT_SOLVER_OPTIONS_ROBUST
    accepted = ("optimal",) if len(Ls) == 1 else ("optimal", "optimal_inaccurate")
    if cap is not None:
        # feasibility first: the smallest one-way turnover the other constraints allow, from a linear program. A cap below
        # it would hand the solver a problem with no solution, and on such a problem the solver's status is not reliable
        # (notebook 11, check D5); so the cap is raised to that turnover, with a margin of one part in a million plus tol,
        # and the month is marked relaxed, before the capped problem is posed.
        w_t, prob_t = _build(Ls, b, lower, upper, w0, None, objective="turnover", betas=betas)
        _run(prob_t, solver, options)
        if prob_t.status not in ("optimal", "optimal_inaccurate"):
            raise OptimiserError(f"infeasible without the turnover cap: {prob_t.status}")
        check(_clean(w_t.value), b, lower, upper, w0, None, tol)      # the program's own weights pass the checks too
        if float(prob_t.value) > cap + 10 * tol:
            cap = float(prob_t.value) * (1.0 + 1e-6) + tol
            relaxed = True
    w, prob = _build(Ls, b, lower, upper, w0, cap, betas=betas)
    _run(prob, solver, options)
    x = _clean(w.value)
    if prob.status not in accepted:
        raise OptimiserError(f"solver status {prob.status} on a problem known to be feasible")
    check(x, b, lower, upper, w0, cap, tol)
    a = x - b
    binding = {}
    if "long_only" in constraints:
        binding["long_only"] = int(((x <= tol) & ~exclude_mask).sum())
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        binding["active_weight_bound"] = int((untilted & (np.abs(np.abs(a) - active_bound) <= tol)).sum())
    if "tilt" in constraints:
        binding["tilt"] = int((tilt_mask & (np.abs(x - tilt_share * b) <= tol)).sum())
    to = one_way_turnover(x, w0) if w0 is not None else float("nan")
    if "turnover_cap" in constraints:
        binding["turnover_cap"] = int(abs(to - cap) <= 10 * tol)
    for beta in betas or []:
        if abs(beta @ a) > 10 * tol:
            raise OptimiserError(f"the portfolio's beta differs from the benchmark's by {beta @ a:.2e}")
    te = max(forecast_tracking_error(x, b, S) for S in Sigmas)      # the worst case across the covariances; one covariance, its own
    return Solution(weights=x, active=a, tracking_error=te, turnover=to,
                    turnover_cap_used=cap, relaxed=relaxed, status=prob.status, binding=binding)


def _run(prob, solver: str, options: dict) -> None:
    """Solve with the tight stopping rule; if the solver reports the result as inaccurate at that rule, solve again at
    its default rule (about 1e-8). A single-covariance problem is accepted only when reported optimal; the robust
    problem, written with second-order cones, is accepted as inaccurate too once the feasibility checks pass, and
    the status is recorded so that the evaluation can count such months."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Solution may be inaccurate")   # that case is solved again below at the default rule
        prob.solve(solver=solver, **options)
        if prob.status == "optimal_inaccurate":
            prob.solve(solver=solver)


def _clean(value) -> np.ndarray:
    if value is None:
        return np.full(0, np.nan)
    x = np.asarray(value, dtype=float).ravel()
    return np.where(np.abs(x) < 1e-12, 0.0, x)


def feasible(w: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None, tol: float) -> bool:
    """True when the weights sum to one, lie inside their bounds and respect the turnover cap, all within tol."""
    if len(w) != len(b) or not np.isfinite(w).all() or abs(w.sum() - 1.0) > tol:
        return False
    if (w < lower - tol).any() or (w > upper + tol).any():
        return False
    if cap is not None and w0 is not None and one_way_turnover(w, w0) > cap + 10 * tol:
        return False
    return True


def check(w: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None, tol: float) -> None:
    """Raise unless the weights sum to one, lie inside their bounds and respect the turnover cap, all within tol."""
    if len(w) != len(b) or not np.isfinite(w).all():
        raise OptimiserError("the solver returned no weights")
    if abs(w.sum() - 1.0) > tol:
        raise OptimiserError(f"weights sum to {w.sum():.8f}")
    if (w < lower - tol).any() or (w > upper + tol).any():
        raise OptimiserError("a weight lies outside its bounds")
    if cap is not None and w0 is not None and one_way_turnover(w, w0) > cap + 10 * tol:
        raise OptimiserError(f"turnover {one_way_turnover(w, w0):.6f} exceeds the cap {cap:.6f}")


def masks(columns, tilt=C.TILT_INDUSTRIES, exclude=()) -> tuple[np.ndarray, np.ndarray]:
    """Boolean masks over `columns` for the tilted and the excluded industries; a name that is not a column raises."""
    cols = [str(c).strip() for c in columns]
    for name in tuple(tilt) + tuple(exclude):
        if name not in cols:
            raise KeyError(f"{name} is not one of the {len(cols)} industries")
    return np.array([c in tilt for c in cols]), np.array([c in exclude for c in cols])


# ---------------------------------------------------------------------------
# Version 2, module A: the factor objective under the mandate (notebook 14)
# ---------------------------------------------------------------------------
# A second problem beside the first, which stays as it is. For a month with
# benchmark weights b, a monthly covariance Sigma, the industries' betas B to
# the six factors (N by K, estimated before the month as in notebook 13) and
# the target vector s (constants.FO_TARGET_SIGN), the optimiser finds the
# weights w that maximise the targeted exposure s' B' (w - b) under every
# constraint of the named set and the budget: the forecast tracking error
# sqrt(12 (w - b)' Sigma (w - b)) at most tau. The objective is linear and the
# budget a second-order cone, so the problem is a convex program with one
# global maximum, solved by cvxpy with the same solver as the first problem.
# In a month in which the budget cannot be met (the tracking-error-minimising
# portfolio of the first problem already exceeds tau), the path takes that
# portfolio, the month is marked and counted, and nothing else is relaxed
# (constants.FO_INFEASIBLE_RULE). Every returned portfolio is checked against
# its bounds, the turnover cap and the budget, whatever the solver's status.

@dataclass
class FactorSolution(Solution):
    budget: float = float("nan")              # tau, per year
    budget_met: bool = True                   # False in a month that took the tracking-error-minimising portfolio
    objective: float = float("nan")           # s' B' (w - b), the targeted exposure the month bought
    exposures: np.ndarray = field(default_factory=lambda: np.full(0, np.nan))   # B' (w - b), one per factor


def _run_factor(prob, solver: str, options: dict) -> None:
    """As `_run`, with a third pass at the looser rule of constants.FO_SOLVER_OPTIONS_LOOSE when the solver fails
    outright at the first two, and no exception when the third fails as well (added 18 October 2026, before notebook
    15's first recorded run). Two cases were met while the notebook was built: a month in which the budget cone and
    the turnover cap are both active, where the interior-point method stalls at a residual of 1e-7 and then fails at
    the default rule, which the third pass solves; and a month in which the tracking-error minimiser exceeds the
    budget by a hundredth of a basis point, so that the factor problem has no interior and the solver reports a
    numerical error at every rule, which the caller resolves by the infeasibility rule (the month takes the
    minimiser's portfolio, after the check that the minimiser does exceed the budget). Whatever the pass, a returned
    portfolio is accepted only after the checks on the bounds, the cap and the budget, and the month keeps the
    solver's status word."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Solution may be inaccurate")
        try:
            prob.solve(solver=solver, **options)
            if prob.status == "optimal_inaccurate":
                prob.solve(solver=solver)
        except cp.error.SolverError:
            try:
                prob.solve(solver=solver, **C.FO_SOLVER_OPTIONS_LOOSE)
            except cp.error.SolverError:
                pass                       # the status and the weights stay as the last pass left them; the caller's checks decide


def targeted_exposure(B: np.ndarray, s: np.ndarray, w: np.ndarray, b: np.ndarray) -> float:
    """s' B' (w - b): the sum of the active exposures to the targeted factors, each weighted by its sign in s."""
    return float(np.asarray(s, dtype=float) @ np.asarray(B, dtype=float).T @ (np.asarray(w, dtype=float) - np.asarray(b, dtype=float)))


def _build_factor(Ls: list, b: np.ndarray, g: np.ndarray, tau_m: float, lower: np.ndarray, upper: np.ndarray,
                  w0: np.ndarray | None, cap: float | None, betas: list | None):
    """The cvxpy problem of the factor objective: maximise g' (w - b) with g = B s, under the bounds, the beta
    equalities, the turnover cap written as buys and sells, and one budget cone per covariance factor in `Ls`
    (one for the single-covariance problem, five for the robust one). Written in percent, as the robust problem is."""
    N = len(b)
    w = cp.Variable(N)
    cons = [cp.sum(w) == 1]
    finite_lo = np.isfinite(lower)
    finite_up = np.isfinite(upper)
    if finite_lo.any():
        cons.append(w[finite_lo] >= lower[finite_lo])
    if finite_up.any():
        cons.append(w[finite_up] <= upper[finite_up])
    for beta in betas or []:
        cons.append(beta @ (w - b) == 0)
    if w0 is not None:
        p = cp.Variable(N, nonneg=True)
        n = cp.Variable(N, nonneg=True)
        cons.append(w - w0 == p - n)
        if cap is not None:
            cons.append(0.5 * cp.sum(p + n) <= cap)
    cons += [cp.norm(100.0 * (L.T @ (w - b)), 2) <= 100.0 * tau_m for L in Ls]
    return w, cp.Problem(cp.Maximize(g @ (w - b)), cons)


def solve_factor_objective(Sigma, b: np.ndarray, B: np.ndarray, s: np.ndarray, budget_annual: float, constraints: tuple,
                           w_prev: np.ndarray | None = None, tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
                           turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
                           tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> FactorSolution:
    """The factor objective under the mandate for one month.

    `Sigma` is one monthly covariance, or a list of them for the robust version, in which the budget holds under
    every covariance at once and beta neutrality under the average of their beta vectors. `B` is N by K, `s` the
    K signs, `budget_annual` tau per year. If the turnover cap alone makes the month infeasible it is raised to the
    smallest feasible turnover, as in the first problem. If the budget cannot be met, the month takes the
    tracking-error-minimising portfolio of the first problem and `budget_met` is False.
    """
    Sigmas = [np.asarray(S, dtype=float) for S in (Sigma if isinstance(Sigma, (list, tuple)) else [Sigma])]
    b = np.asarray(b, dtype=float)
    B = np.asarray(B, dtype=float)
    s = np.asarray(s, dtype=float)
    N = len(b)
    if B.shape[0] != N or B.shape[1] != len(s):
        raise OptimiserError(f"the betas are {B.shape} for {N} industries and {len(s)} targets")
    if not budget_annual > 0:
        raise OptimiserError("the budget must be positive")
    tilt_mask = np.zeros(N, bool) if tilt_mask is None else np.asarray(tilt_mask, bool)
    exclude_mask = np.zeros(N, bool) if exclude_mask is None else np.asarray(exclude_mask, bool)
    if "turnover_cap" in constraints and w_prev is None:
        raise OptimiserError("the turnover cap needs the previous portfolio")
    if "exclusion" in constraints and not exclude_mask.any():
        raise OptimiserError("the exclusion constraint needs a mask naming at least one industry")
    lower, upper = bounds(b, constraints, tilt_mask, exclude_mask, active_bound, tilt_share)
    if (lower > upper + tol).any():
        raise OptimiserError("a weight's lower bound exceeds its upper bound; the constraint set is contradictory for this month")
    Ls = [cholesky_factor(S) for S in Sigmas]
    w0 = None if w_prev is None else np.asarray(w_prev, dtype=float)
    cap = turnover_cap if "turnover_cap" in constraints else None
    betas = [np.mean([benchmark_betas(S, b) for S in Sigmas], axis=0)] if "beta_neutral" in constraints else None
    g = B @ s
    tau_m = float(budget_annual) / np.sqrt(PERIODS_PER_YEAR)
    relaxed = False
    options = C.OPT_SOLVER_OPTIONS_ROBUST            # the budget is a cone, so the robust problem's stopping rule applies
    if cap is not None:
        # feasibility first, as in the first problem: the smallest one-way turnover the other constraints allow
        w_t, prob_t = _build(Ls, b, lower, upper, w0, None, objective="turnover", betas=betas)
        _run_factor(prob_t, solver, options)
        if prob_t.status not in ("optimal", "optimal_inaccurate"):
            raise OptimiserError(f"infeasible without the turnover cap: {prob_t.status}")
        check(_clean(w_t.value), b, lower, upper, w0, None, tol)
        if float(prob_t.value) > cap + 10 * tol:
            cap = float(prob_t.value) * (1.0 + 1e-6) + tol
            relaxed = True
    w, prob = _build_factor(Ls, b, g, tau_m, lower, upper, w0, cap, betas)
    _run_factor(prob, solver, options)
    x = _clean(w.value)
    met = prob.status in ("optimal", "optimal_inaccurate") and len(x) == N and np.isfinite(x).all()
    if met:
        met = feasible(x, b, lower, upper, w0, cap, tol) and max(forecast_tracking_error(x, b, S) for S in Sigmas) <= budget_annual * (1.0 + 1e-6) + 10 * tol
    if not met:
        # the budget cannot be met this month: the tracking-error-minimising portfolio of the first problem, under the same
        # constraints and the same cap, is the month's portfolio; its forecast must indeed exceed the budget
        base = _solve(Sigmas, b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)
        if base.tracking_error <= budget_annual * (1.0 - 1e-6) - 10 * tol:
            raise OptimiserError(f"the solver reported {prob.status} on a budget of {budget_annual:.4f} that the minimiser meets at {base.tracking_error:.4f}")
        x, cap, relaxed, status = base.weights, base.turnover_cap_used, base.relaxed, base.status
    else:
        status = prob.status
    check(x, b, lower, upper, w0, cap, tol)
    a = x - b
    for beta in betas or []:
        if abs(beta @ a) > 10 * tol:
            raise OptimiserError(f"the portfolio's beta differs from the benchmark's by {beta @ a:.2e}")
    binding = {}
    if "long_only" in constraints:
        binding["long_only"] = int(((x <= tol) & ~exclude_mask).sum())
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        binding["active_weight_bound"] = int((untilted & (np.abs(np.abs(a) - active_bound) <= tol)).sum())
    if "tilt" in constraints:
        binding["tilt"] = int((tilt_mask & (np.abs(x - tilt_share * b) <= tol)).sum())
    to = one_way_turnover(x, w0) if w0 is not None else float("nan")
    if "turnover_cap" in constraints:
        binding["turnover_cap"] = int(abs(to - cap) <= 10 * tol)
    te = max(forecast_tracking_error(x, b, S) for S in Sigmas)
    binding["budget"] = int(met and abs(te - budget_annual) <= C.FO_E1_BIND_TOL_ANNUAL)
    return FactorSolution(weights=x, active=a, tracking_error=te, turnover=to, turnover_cap_used=cap, relaxed=relaxed, status=status,
                          binding=binding, budget=float(budget_annual), budget_met=bool(met), objective=float(g @ a), exposures=B.T @ a)

In [ ]:
%%writefile src/bp/evaluation.py
"""The rolling evaluation of the extension (notebook 12): one portfolio path per specification, month by month.

A path is one variant (a covariance estimator, or the robust portfolio built from all of them) under one
constraint set. It starts as the index. In every evaluation month the covariance is estimated on the data before
the month, the optimiser solves for the weights with the drifted previous weights as the turnover reference, the
portfolio and the benchmark earn the month's returns, and the active return, the forecast tracking error, the
turnover and the state of the cap are recorded. The measures of a path are computed from its records: realised
tracking error, forecast tracking error and its calibration, active return gross and net of trading costs,
turnover and relaxed-cap months, holdings.

Added 18 October 2026 (constants.EVAL_ADDITIONS_DATE): two descriptive figures of a path, the worst cumulative fall
of the active return and the share of positive months, and a block-bootstrap band on the bias statistic. Version 2,
module A (notebook 15): the factor path, which each month maximises the targeted exposure under the mandate and a
budget on the forecast tracking error, and its measures.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import optimiser as O

PERIODS_PER_YEAR = 12


def run_path(months, Sigmas, weights: pd.DataFrame, returns: pd.DataFrame, constraints: tuple,
             tilt_mask: np.ndarray, exclude_mask: np.ndarray | None, robust: bool = False, keep_weights: bool = False) -> pd.DataFrame:
    """One path. `Sigmas` maps a month to its covariance (or to the list of covariances when `robust`).

    Returns one record per month: forecast tracking error (per year), active return, one-way turnover, the cap
    used and whether it was relaxed, the effective number of holdings, the industries held, and the weights when
    asked. The previous portfolio of the first month is the benchmark of that month (constants.EVAL_START_PORTFOLIO).
    """
    records = []
    w_prev = None
    prev_month = None
    for month in months:
        b = weights.loc[month].to_numpy()
        r = returns.loc[month].to_numpy()
        if w_prev is None:
            w0 = b.copy()
            b_drift = b.copy()
        else:
            w0 = O.drifted_weights(w_prev, returns.loc[prev_month].to_numpy())
            b_drift = O.drifted_weights(weights.loc[prev_month].to_numpy(), returns.loc[prev_month].to_numpy())
        kwargs = dict(w_prev=w0, tilt_mask=tilt_mask, exclude_mask=exclude_mask)
        sol = O.solve_robust(Sigmas[month], b, constraints, **kwargs) if robust else O.solve(Sigmas[month], b, constraints, **kwargs)
        rec = {"month": month, "forecast_te": sol.tracking_error, "active_return": float(sol.weights @ r - b @ r),
               "turnover": sol.turnover, "cap_used": sol.turnover_cap_used if "turnover_cap" in constraints else np.nan,
               "relaxed": bool(sol.relaxed), "effective_number": O.effective_number(sol.weights), "held": int((sol.weights > 1e-6).sum()),
               "status": sol.status,
               "benchmark_turnover": O.one_way_turnover(b, b_drift)}       # the index's own turnover: how far its weights moved beyond what its returns did
        if keep_weights:
            rec["weights"] = sol.weights
        records.append(rec)
        w_prev, prev_month = sol.weights, month
    return pd.DataFrame(records).set_index("month")


def realised_tracking_error(active: pd.Series | np.ndarray) -> float:
    """The standard deviation of the monthly active returns, times the square root of 12."""
    return float(np.std(np.asarray(active, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


def bias_statistic(active: np.ndarray, forecast_te: np.ndarray, scale: np.ndarray | None = None) -> float:
    """sd(active return / forecast monthly standard deviation); `scale` multiplies the forecast variance (the variance-ratio
    correction). nan when a forecast is zero, as under the sets in which the portfolio is the benchmark."""
    f = np.asarray(forecast_te, dtype=float) / np.sqrt(PERIODS_PER_YEAR)
    if scale is not None:
        f = f * np.sqrt(np.asarray(scale, dtype=float))
    if (np.asarray(forecast_te, dtype=float) <= C.OPT_TOL_TE).any():     # a forecast of zero (the portfolio is the benchmark) has no calibration
        return float("nan")
    return float(np.std(np.asarray(active, dtype=float) / f, ddof=1))


def summarise(path: pd.DataFrame, vr_scale: pd.Series | None = None) -> dict:
    """The measures of one path, in return units per year where they are returns or risks."""
    active = path["active_return"].to_numpy()
    te = realised_tracking_error(active)
    gross = PERIODS_PER_YEAR * float(np.mean(active))
    turnover = float(path["turnover"].mean())
    out = {"realised TE": te, "forecast TE (mean)": float(path["forecast_te"].mean()),
           "bias": bias_statistic(active, path["forecast_te"].to_numpy()),
           "bias (variance ratio)": bias_statistic(active, path["forecast_te"].to_numpy(), None if vr_scale is None else vr_scale.loc[path.index].to_numpy()),
           "active return (gross)": gross, "turnover (one-way)": turnover, "benchmark turnover": float(path["benchmark_turnover"].mean()),
           "relaxed months": int(path["relaxed"].sum()), "relaxed share": float(path["relaxed"].mean()),
           "inaccurate months": int((path["status"] != "optimal").sum()),
           "effective number": float(path["effective_number"].mean()), "industries held": float(path["held"].mean())}
    for c in C.EXT_COST_LEVELS:
        bp = int(round(c * 1e4))
        cost = PERIODS_PER_YEAR * turnover * c
        out[f"cost at {bp} bp"] = cost
        out[f"net active return at {bp} bp"] = gross - cost
        out[f"information ratio at {bp} bp"] = (gross - cost) / te if te > C.OPT_TOL_TE else float("nan")   # no ratio for a portfolio that is the benchmark
    return out


def by_decade(path: pd.DataFrame, first_label: str, last_label: str) -> pd.Series:
    """Realised tracking error by decade of the evaluation month; the first and last partial decades carry the given labels."""
    def label(m):
        if m.year < 1990:
            return first_label
        if m.year >= 2020:
            return last_label
        return f"{(m.year // 10) * 10}s"
    groups = pd.Series([label(m) for m in path.index], index=path.index)
    return path.groupby(groups)["active_return"].apply(realised_tracking_error)


def calibrated_forecast(path: pd.DataFrame, window: int = C.EVAL_CALIBRATION_WINDOW, minimum: int = C.EVAL_CALIBRATION_MIN) -> pd.Series:
    """The forecast tracking error of month t multiplied by the ratio of realised to forecast tracking error over the
    previous `window` months of the same path (square root of the mean squared active return over the mean forecast
    variance), using only months before t; unchanged while fewer than `minimum` months are available."""
    f_var = (path["forecast_te"] / np.sqrt(PERIODS_PER_YEAR)) ** 2
    a2 = path["active_return"] ** 2
    num = a2.shift(1).rolling(window, min_periods=minimum).mean()
    den = f_var.shift(1).rolling(window, min_periods=minimum).mean()
    ratio = np.sqrt(num / den)
    ratio = ratio.where(den > 0).fillna(1.0)
    return path["forecast_te"] * ratio


# ---------------------------------------------------------------------------
# Added 18 October 2026: two descriptive figures and the block-bootstrap band
# ---------------------------------------------------------------------------

def worst_cumulative_fall(active: pd.Series) -> dict:
    """The largest fall of the cumulative active return (the running sum of the monthly active returns, starting
    at zero before the first month) from an earlier high: its size in return units, the month of the high, the
    month of the low and the months between them. A descriptive figure, in no claim."""
    a = pd.Series(np.asarray(active, dtype=float), index=active.index if isinstance(active, pd.Series) else None)
    c = a.cumsum()
    high = np.maximum.accumulate(np.concatenate([[0.0], c.to_numpy()]))[1:]
    fall = pd.Series(high - c.to_numpy(), index=c.index)
    if fall.max() <= 0:
        return {"fall": 0.0, "peak": None, "trough": None, "months": 0}
    trough = fall.idxmax()
    before = c.loc[:trough]
    peak = before.idxmax() if before.max() > 0 else before.index[0]       # a fall from the start has no earlier high; the first month stands for it
    months = int(c.index.get_loc(trough) - c.index.get_loc(peak)) + (0 if before.max() > 0 else 1)
    return {"fall": float(fall.max()), "peak": peak if before.max() > 0 else "start", "trough": trough, "months": months}


def rolling_tracking_error(active: pd.DataFrame, window: int = C.EVAL_ROLLING_TE_MONTHS) -> pd.DataFrame:
    """The rolling realised tracking error of every column: the standard deviation (n minus 1) of the monthly
    active return over the `window` months ending in and including each month, times the square root of 12,
    from the first month with a full window. Added 6 November 2026 (constants.EVAL_ROLLING_ADDITIONS_DATE)."""
    if window < 2 or window > len(active):
        raise ValueError(f"the window must be between 2 and the number of months, {len(active)}; got {window}")
    return active.rolling(window).std(ddof=1).iloc[window - 1:] * np.sqrt(PERIODS_PER_YEAR)


def share_positive(active: pd.Series | np.ndarray) -> float:
    """The share of months in which the active return is above zero."""
    a = np.asarray(active, dtype=float)
    return float(np.mean(a > 0))


def block_bootstrap_band(active: np.ndarray, forecast_te: np.ndarray, block: int = C.EVAL_BOOTSTRAP_BLOCK_MONTHS,
                         draws: int = C.EVAL_BOOTSTRAP_DRAWS, seed: int = C.EVAL_BOOTSTRAP_SEED, level: float = C.EVAL_BOOTSTRAP_LEVEL) -> tuple[float, float]:
    """A band on the bias statistic from a circular block bootstrap: each resample draws ceil(n / block) starting
    months at random and takes `block` consecutive months from each (running past the last month into the first),
    keeping the month's pair (active return, forecast) together; the band is the central `level` share of the
    resampled statistics. Deterministic for a given seed."""
    a = np.asarray(active, dtype=float)
    f = np.asarray(forecast_te, dtype=float) / np.sqrt(PERIODS_PER_YEAR)
    n = len(a)
    if n != len(f) or n < 2 or block < 1 or draws < 1:
        raise ValueError("the band needs at least two months, a block of one month or more and one draw or more")
    if (np.asarray(forecast_te, dtype=float) <= C.OPT_TOL_TE).any():
        return (float("nan"), float("nan"))
    ratio = a / f
    rng = np.random.default_rng(seed)
    k = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(draws, k))
    idx = ((starts[:, :, None] + np.arange(block)[None, None, :]) % n).reshape(draws, k * block)[:, :n]
    stats = np.std(ratio[idx], axis=1, ddof=1)
    tail = 100.0 * (1.0 - level) / 2.0
    lo, hi = np.percentile(stats, [tail, 100.0 - tail])
    return (float(lo), float(hi))


# ---------------------------------------------------------------------------
# Version 2, module A: the factor path (notebook 15)
# ---------------------------------------------------------------------------

def run_factor_path(months, Sigmas, betas: dict, weights: pd.DataFrame, returns: pd.DataFrame, constraints: tuple, s: np.ndarray,
                    budget_annual: float, tilt_mask: np.ndarray, exclude_mask: np.ndarray | None, keep_weights: bool = False) -> pd.DataFrame:
    """One path of the factor objective, run as `run_path` runs a path of version 1: in every month the optimiser
    maximises s' B' (w - b) under the constraints and the budget, with the drifted previous weights as the turnover
    reference, `Sigmas[month]` the month's covariance (a list for the robust variant) and `betas[month]` its N by K
    betas, both estimated before the month. The month in which the budget cannot be met takes the
    tracking-error-minimising portfolio and is marked (`budget_met` False), as constants.FO_INFEASIBLE_RULE says.

    Returns one record per month: the forecast tracking error, the active return, the turnover and the cap, whether
    the budget was met and whether it binds, the objective and the active exposure to each factor, the gap
    s' (B_t - B_{t+1})' a_t once the next month's betas are known (nan in the last month), and the holdings.
    """
    names = list(C.ATTRIBUTION_FACTORS)
    records = []
    w_prev = None
    prev_month = None
    prev = None                                  # (active weights, betas) of the previous month, for its gap
    for month in months:
        b = weights.loc[month].to_numpy()
        r = returns.loc[month].to_numpy()
        B = np.asarray(betas[month], dtype=float)
        if w_prev is None:
            w0 = b.copy()
            b_drift = b.copy()
        else:
            w0 = O.drifted_weights(w_prev, returns.loc[prev_month].to_numpy())
            b_drift = O.drifted_weights(weights.loc[prev_month].to_numpy(), returns.loc[prev_month].to_numpy())
        sol = O.solve_factor_objective(Sigmas[month], b, B, s, budget_annual, constraints, w_prev=w0, tilt_mask=tilt_mask, exclude_mask=exclude_mask)
        if prev is not None:
            a_prev, B_prev = prev
            records[-1]["gap"] = float(np.asarray(s, dtype=float) @ (B_prev - B).T @ a_prev)
        rec = {"month": month, "forecast_te": sol.tracking_error, "active_return": float(sol.weights @ r - b @ r),
               "turnover": sol.turnover, "cap_used": sol.turnover_cap_used if "turnover_cap" in constraints else np.nan,
               "relaxed": bool(sol.relaxed), "cap_binds": bool(sol.binding.get("turnover_cap", 0)),
               "budget_met": bool(sol.budget_met), "budget_binds": bool(sol.binding["budget"]), "objective": sol.objective,
               **{f"exposure {f}": float(x) for f, x in zip(names, sol.exposures)},
               "gap": np.nan, "effective_number": O.effective_number(sol.weights), "held": int((sol.weights > 1e-6).sum()),
               "status": sol.status, "benchmark_turnover": O.one_way_turnover(b, b_drift)}
        if keep_weights:
            rec["weights"] = sol.weights
        records.append(rec)
        prev = (sol.active, B)
        w_prev, prev_month = sol.weights, month
    return pd.DataFrame(records).set_index("month")


def summarise_factor(path: pd.DataFrame, budget_annual: float, targeted: tuple = C.FO_TARGET_FACTORS) -> dict:
    """The measures of one factor path: version 1's measures (realised and forecast tracking error, the bias
    statistic, active return gross and net of costs, turnover, the cap) and module A's (realised over budget, the
    months the budget was not met and the months it binds, the mean active exposure to each targeted factor and
    their sum, the exposure bought per basis point of realised tracking error, the mean absolute gap)."""
    out = summarise(path)
    te = out["realised TE"]
    years = len(path) / PERIODS_PER_YEAR
    out["budget"] = float(budget_annual)
    out["realised over budget"] = te / budget_annual
    out["budget not met months"] = int((~path["budget_met"]).sum())
    out["budget binds months"] = int(path["budget_binds"].sum())
    out["cap binds months"] = int(path["cap_binds"].sum())
    for f in targeted:
        out[f"mean exposure {f}"] = float(path[f"exposure {f}"].mean())
    out["mean objective"] = float(path["objective"].mean())
    out["mean objective per targeted factor"] = float(path["objective"].mean()) / len(targeted)
    out["exposure per bp of realised TE"] = float(path["objective"].mean()) / (1e4 * te) if te > C.OPT_TOL_TE else float("nan")
    out["mean absolute gap"] = float(path["gap"].abs().mean())
    out["standard error of the gross active return"] = te / np.sqrt(years)
    return out

In [ ]:
%%writefile src/bp/attribution.py
"""Factor attribution of a path's active return (notebook 13).

For each month t of a path: the active weights a_t = w_t - b_t (portfolio minus
benchmark, one number per industry); the industries' betas B_t to the factors
(49 by K), estimated on the data before t by the same regression as the factor
covariance estimator of notebook 10; the active exposures x_t = B_t' a_t, K
numbers that say how much more or less of each factor the portfolio holds than
the benchmark; the factor contributions x_{t,k} F_{t,k}, with F_t the factors'
returns in month t; and the residual, the active return minus the sum of the
contributions, the part of the active return the factors do not explain (the
industries' own returns). Contributions plus residual equal the active return
by construction; the identity is checked within
constants.ATTRIBUTION_IDENTITY_MAX_ABS_ERROR.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C

PERIODS_PER_YEAR = 12


def active_exposures(B: np.ndarray, w: np.ndarray, b: np.ndarray) -> np.ndarray:
    """x = B' (w - b): the portfolio's factor exposures minus the benchmark's, one per factor."""
    return np.asarray(B, dtype=float).T @ (np.asarray(w, dtype=float) - np.asarray(b, dtype=float))


def attribute(weights: pd.DataFrame, bench_weights: pd.DataFrame, betas: dict, factors: pd.DataFrame,
              active_return: pd.Series, names: tuple = C.ATTRIBUTION_FACTORS) -> pd.DataFrame:
    """One row per month: the active exposure to each factor, each factor's contribution (exposure times the
    factor's return in the month), their total, the residual (active return minus the total) and the identity
    error (total plus residual minus active return, zero up to floating-point arithmetic).

    `weights` holds the path's weights by month (months by N), `bench_weights` the benchmark's for the same months,
    `betas` maps a month to its B (N by K), `factors` holds the K factors' monthly returns, `active_return` the
    path's recorded active return by month."""
    rows = []
    for month in weights.index:
        w = weights.loc[month].to_numpy()
        b = bench_weights.loc[month].to_numpy()
        x = active_exposures(betas[month], w, b)
        f = factors.loc[month, list(names)].to_numpy(dtype=float)
        contrib = x * f
        a = float(active_return.loc[month])
        total = float(contrib.sum())
        row = {"month": month}
        row.update({f"exposure {n}": float(v) for n, v in zip(names, x)})
        row.update({f"contribution {n}": float(v) for n, v in zip(names, contrib)})
        row["factor total"] = total
        row["residual"] = a - total
        row["active return"] = a
        row["identity error"] = total + row["residual"] - a
        rows.append(row)
    return pd.DataFrame(rows).set_index("month")


def summarise(att: pd.DataFrame, names: tuple = C.ATTRIBUTION_FACTORS) -> dict:
    """The attribution of one path over its months: the mean active exposure to each factor; each factor's mean
    contribution per year and its standard error (the standard deviation of the monthly contributions divided by
    the square root of the number of months, times 12); the same for the factor total, the residual and the
    active return; and the factor share, the share of the variance of the active return that the factor total
    explains, 1 minus the variance of the residual over the variance of the active return."""
    T = len(att)
    out = {"months": T}
    for n in names:
        out[f"exposure {n}"] = float(att[f"exposure {n}"].mean())
    for col in [f"contribution {n}" for n in names] + ["factor total", "residual", "active return"]:
        x = att[col].to_numpy(dtype=float)
        out[f"{col} (per year)"] = PERIODS_PER_YEAR * float(x.mean())
        out[f"{col} (standard error)"] = PERIODS_PER_YEAR * float(x.std(ddof=1)) / np.sqrt(T)
    var_a = float(att["active return"].var(ddof=1))
    out["factor share"] = 1.0 - float(att["residual"].var(ddof=1)) / var_a if var_a > 0 else float("nan")
    out["largest identity error"] = float(att["identity error"].abs().max())
    return out


def by_period(att: pd.DataFrame, labels: pd.Series, names: tuple = C.ATTRIBUTION_FACTORS) -> pd.DataFrame:
    """Mean contributions per year by a grouping of the months (for example decades), one row per group:
    each factor, the factor total, the residual and the active return."""
    cols = [f"contribution {n}" for n in names] + ["factor total", "residual", "active return"]
    return PERIODS_PER_YEAR * att[cols].groupby(labels.loc[att.index]).mean()

## Count first: the months, the factors, the nineteen paths

In [ ]:
import sys
sys.path.insert(0, "src")
import json, time
import numpy as np
import pandas as pd
from bp import constants as C
from bp import french_loader as fl
from bp import covariance as CV
from bp import optimiser as O
from bp import evaluation as E
from bp import attribution as A

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)

P = CV.Panels()
weights, returns_m = P.weights, P.returns_m
months = pd.period_range(C.COV_EVAL_START, P.last, freq="M")
industries = [c.strip() for c in weights.columns]
tilt_mask, _ = O.masks(weights.columns)
_, mandate_mask = O.masks(weights.columns, exclude=C.MANDATE_EXCLUSIONS)
FACTORS = list(C.ATTRIBUTION_FACTORS)
TARGETED = list(C.FO_TARGET_FACTORS)
FACTOR_WORD = {"HML": "value", "RMW": "profitability", "CMA": "investment", "Mom": "momentum"}
S_ALL = np.array([C.FO_TARGET_SIGN[f] for f in FACTORS])
MANDATE = C.CONSTRAINT_SETS[C.FO_CONSTRAINT_SET]
VARIANTS = list(CV.ALL_VARIANTS)
LABEL = {v: ("robust, all five" if v == C.ROBUST_VARIANT else ("RiskMetrics, daily 5y" if v == C.EWMA_VARIANT else f"{v[0]}, {v[1].replace('_', ' ')}")) for v in VARIANTS}
SAMPLE = ("sample", "daily_3y")
BP = {tau: f"{1e4 * tau:.0f}" for tau in C.FO_BUDGETS_ANNUAL}
factors_m = P.factors_m.loc[months, FACTORS]
assert factors_m.notna().all().all() and len(factors_m) == len(months), "a factor is missing in an evaluation month"

# the nineteen paths, as notebook 15 names them: (label, variant, budget, target vector, the targeted factors of the path)
PATHS = []
for v in CV.EVALUATION_VARIANTS:
    for tau in C.FO_BUDGETS_ANNUAL:
        PATHS.append((f"{LABEL[v]} | {BP[tau]}", v, tau, S_ALL, TARGETED))
for v in (C.ROBUST_VARIANT, C.EWMA_VARIANT):
    PATHS.append((f"{LABEL[v]} | {BP[C.FO_BUDGET_MAIN]}", v, C.FO_BUDGET_MAIN, S_ALL, TARGETED))
for single in C.FO_SINGLE_FACTOR_PATHS:
    s_one = np.array([1.0 if f in single else 0.0 for f in FACTORS])
    PATHS.append((f"{LABEL[SAMPLE]}, {FACTOR_WORD[single[0]]} alone | {BP[C.FO_BUDGET_MAIN]}", SAMPLE, C.FO_BUDGET_MAIN, s_one, list(single)))
MAIN = [p[0] for p in PATHS[:15]]                                  # the main paths: the five estimators at the three budgets
MAIN_100_150 = [p[0] for p in PATHS[:15] if p[2] >= C.FO_BUDGET_MAIN]
main100 = [p[0] for p in PATHS if p[2] == C.FO_BUDGET_MAIN and p[0] in MAIN]
TARGET_OF = {p[0]: p[4] for p in PATHS}

def betas_at(month):
    X = CV.daily_window(P.excess_d, month, C.FO_BETA_WINDOW_YEARS)
    _, B = CV.factor_cov(X.to_numpy(), P.factors_d.loc[X.index, FACTORS].to_numpy())
    return B

# Count first
assert len(months) == 566, len(months)
assert len(PATHS) == C.FO_PATH_COUNT == 19 and len(set(p[0] for p in PATHS)) == 19 and len(MAIN) == 15 and len(MAIN_100_150) == 10
assert set(C.FO_ATTRIBUTION_REFERENCE_TE) == set(p[0] for p in PATHS), "the reference tracking errors name the nineteen paths"
assert C.FO_NOTEBOOK_16_JUDGES == ("E4",) and C.FO_E4_FACTOR_SHARE_MIN == 0.30
provenance = [fl.download(key) for key in P.files]
fl.write_provenance(provenance)
vintage = provenance[0]["vintage_line"].split("the ")[1].split(" ")[0]
print(f"{len(months)} evaluation months, {months[0]} to {months[-1]}; {len(FACTORS)} factors with returns in every month; {len(PATHS)} paths, {len(MAIN)} of them main, {len(MAIN_100_150)} at {BP[C.FO_BUDGET_MAIN]} bp and above")
print(f"E4: the four targeted contributions sum to a positive amount on every main path, and the factor share is at least {C.FO_E4_FACTOR_SHARE_MIN:.0%} on every main path at {BP[C.FO_BUDGET_MAIN]} and {BP[max(C.FO_BUDGETS_ANNUAL)]} bp")
print(f"the exposure bought per factor: {C.FO_EXPOSURE_BOUGHT_PER_FACTOR}; the gap per unit: {C.FO_GAP_PER_UNIT} (both dated {C.FO_ATTRIBUTION_DATE}, no expectation)")
print(f"CRSP vintage of the eight files: {vintage}")

## The covariances and the betas, once per month

As in notebook 15: each estimator variant once per month, shared by the paths that use it; the betas are the same for every path.

In [ ]:
t0 = time.time()
Sigmas = {v: {} for v in CV.EVALUATION_VARIANTS + (C.EWMA_VARIANT,)}
betas = {}
for i, month in enumerate(months):
    for v in CV.EVALUATION_VARIANTS + (C.EWMA_VARIANT,):
        Sigmas[v][month], _, _ = CV.estimate(*v, month, P)
    betas[month] = betas_at(month)
    if i % 100 == 0:
        print(f"{month}: {time.time() - t0:.0f}s")
Sigmas[C.ROBUST_VARIANT] = {m: [Sigmas[v][m] for v in CV.EVALUATION_VARIANTS] for m in months}
print(f"{len(months)} months of covariances and betas in {time.time() - t0:.0f}s")

## The nineteen paths, rebuilt with their weights

The paths are rebuilt from the modules with the weights of every month kept, which notebook 15 did not save; each path's realised tracking error is checked against notebook 15's, fixed in `constants.py`, within 0.1 basis point a year. Where the repository's `results/` folder is present, the monthly records are compared with the committed table under the rule for reruns: the problem is a cone program, so every record cell reproduces within 1e-8, and the count of months with an inaccurate solver status is reported beside the comparison and is not part of it.

In [ ]:
paths = {}
t0 = time.time()
for label, v, tau, s_vec, targeted in PATHS:
    t1 = time.time()
    paths[label] = E.run_factor_path(months, Sigmas[v], betas, weights, returns_m, MANDATE, s_vec, tau, tilt_mask, mandate_mask, keep_weights=True)
    p = paths[label]
    print(f"{label:<40} {time.time() - t1:5.0f}s  budget not met {int((~p['budget_met']).sum()):3d}  binds {int(p['budget_binds'].sum()):3d}  inaccurate {int((p['status'] != 'optimal').sum()):3d}")
print(f"{len(paths)} paths in {time.time() - t0:.0f}s")
assert all(len(p) == len(months) for p in paths.values())
repro = pd.DataFrame({"realised TE (bp)": {label: 1e4 * E.realised_tracking_error(p["active_return"]) for label, p in paths.items()},
                      "notebook 15 (bp)": {label: 1e4 * C.FO_ATTRIBUTION_REFERENCE_TE[label] for label in paths}})
repro["difference (bp)"] = repro["realised TE (bp)"] - repro["notebook 15 (bp)"]
ok_repro = int((repro["difference (bp)"].abs() <= 1e4 * C.ATTRIBUTION_REPRODUCTION_TOL).sum())
print(repro.to_string(float_format=lambda x: f"{x:.4f}"))
print(f"the rebuilt paths reproduce notebook 15's realised tracking errors within {1e4 * C.ATTRIBUTION_REPRODUCTION_TOL:.1f} bp in {ok_repro} of {len(repro)}: {'holds' if ok_repro == len(repro) else 'FAILS'}")
# the rule for reruns, against notebook 15's committed monthly records where the repository is present
monthly = pd.concat({label: p.drop(columns=["weights", "status"]) for label, p in paths.items()}, names=["path", "month"])
committed = f"results/{vintage}"
committed_monthly = f"{committed}/factor_objective_monthly_{vintage}.csv"
if os.path.exists(committed_monthly):
    old = pd.read_csv(committed_monthly, index_col=[0, 1])
    old.index = pd.MultiIndex.from_arrays([old.index.get_level_values(0), pd.PeriodIndex(old.index.get_level_values(1), freq="M")], names=["path", "month"])
    num = [c for c in old.columns if old[c].dtype.kind in "fi" and c in monthly.columns]
    new_num = monthly[num].astype(float).round(8)
    gaps = (old[num].astype(float) - new_num).abs().groupby(level="path").max().max(axis=1)
    flags = [c for c in old.columns if old[c].dtype.kind == "b" and c in monthly.columns]
    flag_same = (old[flags] == monthly[flags]).all(axis=1).groupby(level="path").all() if flags else pd.Series(True, index=gaps.index)
    rule_ok = int(((gaps <= C.EVAL_RERUN_CONE_MAX_DIFFERENCE + 1e-14) & flag_same).sum())
    print(f"rerun rule, the monthly records against the committed table (records of a cone program, within {C.EVAL_RERUN_CONE_MAX_DIFFERENCE:.0e} per cell): {rule_ok} of {len(gaps)} paths; the largest difference {gaps.max():.1e} ({gaps.idxmax()})")
    for label in gaps.index:
        if gaps[label] > C.EVAL_RERUN_CONE_MAX_DIFFERENCE + 1e-14 or not flag_same[label]:
            print(f"  {label}: largest difference {gaps[label]:.1e}, state columns {'equal' if flag_same[label] else 'DIFFER'}")
    print("inaccurate solver status per path, this run (reported beside the comparison, not part of it): " + ", ".join(f"{label.split(' | ')[0]} at {label.split(' | ')[1]}: {int((p['status'] != 'optimal').sum())}" for label, p in paths.items()))
else:
    print(f"rerun rule: no {committed}/ folder beside this notebook, so the comparison with the committed records runs where the repository is present")

## The attribution, path by path

One row per path, exactly as notebook 13 computes it: the mean active exposure to each factor over the 566 months; each factor's contribution, the factor total, the residual and the active return in basis points a year with their standard errors; the factor share. The targeted total is the sum of the four targeted factors' contributions, the quantity E4 asks about.

In [ ]:
atts, rows = {}, []
bench_w = weights.loc[months]
for label, v, tau, s_vec, targeted in PATHS:
    path = paths[label]
    W = pd.DataFrame(np.vstack(path["weights"].to_list()), index=path.index, columns=weights.columns)
    att = A.attribute(W, bench_w, betas, factors_m, path["active_return"])
    att["targeted total"] = att[[f"contribution {f}" for f in TARGETED]].sum(axis=1)
    atts[label] = att
    s = A.summarise(att)
    tt = att["targeted total"].to_numpy(dtype=float)
    s["targeted total (per year)"] = 12 * float(tt.mean())
    s["targeted total (standard error)"] = 12 * float(tt.std(ddof=1)) / np.sqrt(len(tt))
    # the decomposition behind the factor share: the factor total's own variance, the residual's and twice their
    # covariance, each over the active return's variance; the three sum to one and the factor share is one minus the second
    a_var = float(att["active return"].var(ddof=1))
    s["factor total variance share"] = float(att["factor total"].var(ddof=1)) / a_var
    s["residual variance share"] = float(att["residual"].var(ddof=1)) / a_var
    s["covariance share"] = 1.0 - s["factor total variance share"] - s["residual variance share"]
    rows.append({"path": label, "variant": LABEL[v], "budget (bp)": int(BP[tau]), "target": ", ".join(FACTOR_WORD[f] for f in targeted), **s})
summary = pd.DataFrame(rows).set_index("path")
largest_identity = summary["largest identity error"].max()
print(f"the identity: contributions plus residual equal the active return within {largest_identity:.1e} in every month of every path (level {C.ATTRIBUTION_IDENTITY_MAX_ABS_ERROR:g}): {'holds' if largest_identity <= C.ATTRIBUTION_IDENTITY_MAX_ABS_ERROR else 'FAILS'}")
print()
exp_cols = [f"exposure {f}" for f in FACTORS]
print("mean active exposures:")
print(summary[exp_cols].rename(columns=lambda c: c.replace("exposure ", "")).to_string(float_format=lambda x: f"{x:+.4f}"))
print()
con_cols = [f"contribution {f} (per year)" for f in FACTORS] + ["targeted total (per year)", "factor total (per year)", "residual (per year)", "active return (per year)"]
se_cols = [c.replace("(per year)", "(standard error)") for c in con_cols]
shown = (1e4 * summary[con_cols]).rename(columns=lambda c: c.replace("contribution ", "").replace(" (per year)", ""))
shown["factor share"] = summary["factor share"]
print("contributions, the targeted total, the factor total, the residual and the active return, basis points a year, and the factor share:")
print(shown.to_string(float_format=lambda x: f"{x:+.2f}"))
print()
print("standard errors of the same, basis points a year:")
print((1e4 * summary[se_cols]).rename(columns=lambda c: c.replace("contribution ", "").replace(" (standard error)", "")).to_string(float_format=lambda x: f"{x:.2f}"))

## E4: the targeted contributions and the factor share

In [ ]:
tt = summary.loc[MAIN, "targeted total (per year)"]
positive4 = int((tt > 0).sum())
share = summary.loc[MAIN_100_150, "factor share"]
above4 = int((share >= C.FO_E4_FACTOR_SHARE_MIN).sum())
ok4 = positive4 == len(MAIN) and above4 == len(MAIN_100_150)
print("the targeted total, basis points a year, with its standard error, and the factor share, main paths:")
e4 = pd.DataFrame({"targeted total (bp a year)": 1e4 * summary.loc[MAIN, "targeted total (per year)"], "standard error (bp)": 1e4 * summary.loc[MAIN, "targeted total (standard error)"],
                   "factor share": summary.loc[MAIN, "factor share"], "active return (bp a year)": 1e4 * summary.loc[MAIN, "active return (per year)"], "standard error of the active return (bp)": 1e4 * summary.loc[MAIN, "active return (standard error)"]})
print(e4.to_string(float_format=lambda x: f"{x:+.2f}"))
print()
print(f"E4: the targeted total is positive on {positive4} of {len(MAIN)} main paths ({1e4 * tt.min():+.1f} to {1e4 * tt.max():+.1f} bp a year); the factor share at {BP[C.FO_BUDGET_MAIN]} and {BP[max(C.FO_BUDGETS_ANNUAL)]} bp is at least {C.FO_E4_FACTOR_SHARE_MIN:.0%} on {above4} of {len(MAIN_100_150)} ({share.min():.2f} to {share.max():.2f}): {'MET' if ok4 else 'NOT MET'}")
share75 = summary.loc[[p for p in MAIN if p not in MAIN_100_150], "factor share"]
print(f"reported, outside E4: the factor share at 75 bp runs from {share75.min():.2f} to {share75.max():.2f}; on the four sensitivity paths " + ", ".join(f"{label.split(' | ')[0]} {summary.loc[label, 'factor share']:.2f}" for label in summary.index if label not in MAIN))
print()
dec = summary.loc[MAIN_100_150, ["factor total variance share", "residual variance share", "covariance share", "factor share"]]
print("the decomposition of the active return's variance, main paths at 100 and 150 bp: the factor total's own variance, the residual's and twice their covariance, each over the active return's variance (the three sum to one; the factor share is one minus the residual's):")
print(dec.to_string(float_format=lambda x: f"{x:+.4f}"))
print(f"the factor total's own variance is {dec['factor total variance share'].min():.0%} to {dec['factor total variance share'].max():.0%} of the active return's, the residual's {dec['residual variance share'].min():.0%} to {dec['residual variance share'].max():.0%}, and the covariance term takes {-dec['covariance share'].max():.0%} to {-dec['covariance share'].min():.0%} off their sum")
ratio = (summary["active return (per year)"] / summary["active return (standard error)"]).abs()
print(f"the active return against its standard error, all nineteen paths: the largest ratio is {ratio.max():.2f} ({ratio.idxmax()}); {int((ratio > 2).sum())} of {len(ratio)} beyond two standard errors")

## The exposure bought, factor by factor

The mean active exposure to each targeted factor divided by the path's realised tracking error in basis points a year; the four sum to notebook 15's exposure per basis point. Reported, no expectation.

In [ ]:
te_bp = pd.Series({label: 1e4 * E.realised_tracking_error(p["active_return"]) for label, p in paths.items()})
bought = pd.DataFrame({FACTOR_WORD[f]: summary[f"exposure {f}"] / te_bp for f in TARGETED})
bought["total"] = bought.sum(axis=1)
bought.insert(0, "realised TE (bp)", te_bp)
for f in TARGETED:
    bought[f"mean exposure {FACTOR_WORD[f]}"] = summary[f"exposure {f}"]
print("the exposure bought per basis point of realised tracking error, by factor (beta units per bp a year), and the mean exposures behind it:")
print(bought.to_string(float_format=lambda x: f"{x:.5f}"))
b100 = bought.loc[main100]
print()
print(f"at {BP[C.FO_BUDGET_MAIN]} bp on the five estimators the total runs from {b100['total'].min():.5f} to {b100['total'].max():.5f}; by factor, the largest share of the total goes to " + ", ".join(f"{FACTOR_WORD[f]} on {int((b100[[FACTOR_WORD[g] for g in TARGETED]].idxmax(axis=1) == FACTOR_WORD[f]).sum())}" for f in TARGETED) + " of 5 paths")

## The gap, and the gap per unit of exposure bought

Notebook 15 recorded, for every month, the factor objective chosen with the betas estimated before the month minus the value the same active weights carry with the betas estimated after it. Here, for every path: the mean absolute gap, the mean signed gap, the largest absolute gap with its month, and the gap per unit, the mean absolute gap divided by the mean factor objective. Reported, no expectation.

In [ ]:
rows = []
for label, p in paths.items():
    g = p["gap"].dropna()
    obj = float(p["objective"].mean())
    rows.append({"path": label, "mean absolute gap": float(g.abs().mean()), "mean gap": float(g.mean()), "largest absolute gap": float(g.abs().max()), "month of the largest": str(g.abs().idxmax()),
                 "mean objective": obj, "gap per unit": float(g.abs().mean()) / obj, "months": int(len(g))})
gap_table = pd.DataFrame(rows).set_index("path")
print("the gap, in beta units, and per unit of the mean factor objective:")
print(gap_table.to_string(float_format=lambda x: f"{x:.5f}"))
val, mom = f"{LABEL[SAMPLE]}, value alone | {BP[C.FO_BUDGET_MAIN]}", f"{LABEL[SAMPLE]}, momentum alone | {BP[C.FO_BUDGET_MAIN]}"
print()
print(f"the two single-factor paths: value alone gap {gap_table.loc[val, 'mean absolute gap']:.5f} on an objective of {gap_table.loc[val, 'mean objective']:.5f} ({gap_table.loc[val, 'gap per unit']:.3f} per unit); momentum alone {gap_table.loc[mom, 'mean absolute gap']:.5f} on {gap_table.loc[mom, 'mean objective']:.5f} ({gap_table.loc[mom, 'gap per unit']:.3f} per unit)")
print(f"on the five estimators at {BP[C.FO_BUDGET_MAIN]} bp the gap per unit runs from {gap_table.loc[main100, 'gap per unit'].min():.3f} to {gap_table.loc[main100, 'gap per unit'].max():.3f}; across the nineteen paths {gap_table['gap per unit'].min():.3f} to {gap_table['gap per unit'].max():.3f}")

## By decade: the five estimators at the main budget

Contributions, the targeted total, the residual and the active return per year for each decade of the evaluation months, for the five estimator paths at 100 basis points; the standard error of a mean over about 120 months is about 2.2 times the one over 566.

In [ ]:
first_label, last_label = f"{months[0]} to 1989-12", f"2020-01 to {months[-1]}"
decade = pd.Series([first_label if m.year < 1990 else (last_label if m.year >= 2020 else f"{(m.year // 10) * 10}s") for m in months], index=months)
blocks = []
for label in main100:
    att = atts[label]
    d = A.by_period(att, decade)
    d["targeted total"] = 12 * att["targeted total"].groupby(decade.loc[att.index]).mean()
    d.insert(0, "path", label)
    blocks.append(d)
decades = pd.concat(blocks).set_index("path", append=True).swaplevel()
print(f"at {BP[C.FO_BUDGET_MAIN]} bp, basis points a year by decade:")
print((1e4 * decades).rename(columns=lambda c: c.replace("contribution ", "")).to_string(float_format=lambda x: f"{x:+.1f}"))

## Save

The attribution table (one row per path, the variance decomposition in its last three columns), the by-decade table, the exposure bought per factor, the gap table and E4's verdict are written to `outputs/` with the vintage in the file name; the solver's name and version join the run's provenance record.

In [ ]:
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
summary.to_csv(f"{C.OUTPUT_DIR}/factor_objective_attribution_{vintage}.csv", float_format="%.8f")
decades.to_csv(f"{C.OUTPUT_DIR}/factor_objective_attribution_by_decade_{vintage}.csv", float_format="%.8f")
bought.to_csv(f"{C.OUTPUT_DIR}/factor_objective_exposure_bought_{vintage}.csv", float_format="%.8f")
gap_table.to_csv(f"{C.OUTPUT_DIR}/factor_objective_gap_{vintage}.csv", float_format="%.8f")
pd.DataFrame([{"expectation": "E4", "verdict": "MET" if ok4 else "NOT MET",
               "detail": f"targeted total positive on {positive4} of {len(MAIN)} main paths; factor share at least {C.FO_E4_FACTOR_SHARE_MIN:.2f} on {above4} of {len(MAIN_100_150)} main paths at {BP[C.FO_BUDGET_MAIN]} and {BP[max(C.FO_BUDGETS_ANNUAL)]} bp ({share.min():.4f} to {share.max():.4f})"}]).set_index("expectation").to_csv(f"{C.OUTPUT_DIR}/factor_objective_attribution_verdict_{vintage}.csv")
# the run's provenance record: the solver's name and version, shared with notebook 12's record when both ran here
import importlib, platform, cvxpy
solver_module = importlib.import_module(C.OPT_SOLVER.lower())
record_path = f"{C.OUTPUT_DIR}/provenance_solver.json"
record = json.load(open(record_path)) if os.path.exists(record_path) else {}
written_by = sorted(set(str(record.get("written_by", "")).replace("notebook ", "").split(", ")) - {""} | {"16"}, key=int)
record.update({"solver": C.OPT_SOLVER, "solver_version": getattr(solver_module, "__version__", "unknown"), "cvxpy": cvxpy.__version__, "numpy": np.__version__, "pandas": pd.__version__, "python": platform.python_version(), "written_by": "notebook " + ", ".join(written_by)})
json.dump(record, open(record_path, "w"), indent=1)
print("solver:", record)
print("written:", sorted(f for f in os.listdir(C.OUTPUT_DIR) if f.startswith(("factor_objective_attribution", "factor_objective_exposure", "factor_objective_gap", "provenance_solver"))))

## What this notebook established, and what could be wrong

| Check or expectation | Result | Verdict |
|---|---|---|
| The rebuilt paths reproduce notebook 15 | realised tracking error within 0.1 basis point a year in 19 of 19 paths; the monthly records within 1e-8 per cell in 19 of 19 under the rule for reruns, with two BLAS threads | holds |
| The identity | contributions plus residual equal the active return within 1.7e-18 in every month of every path | holds |
| E4. The targeted total and the factor share | the four targeted factors' contributions sum to +5.7 to +58.9 basis points a year, positive on 15 of 15 main paths; the factor share at 100 and 150 basis points is 0.14 to 0.28, at or above 0.30 on 0 of 10 | NOT MET |

The rebuilt paths are notebook 15's. Every path's realised tracking error matches the committed one to the fourth decimal of a basis point, and under the rule for reruns every record of the committed monthly table comes back within 1e-8, for all nineteen paths. One month decides whether that last statement holds exactly: in 1982-08 the RiskMetrics path at 100 basis points is at the solver's precision, and the status word it returns depends on the number of threads the linear algebra uses, which notebook 15's closing records from the confirmation run of 6 November; with two threads, as in the recorded run of notebook 15, the month comes out inaccurate and every record reproduces, and with one it comes out optimal and the path's records move by up to 4.6e-7 in 33 months, a difference that reaches no printed digit of this notebook's tables. The identity holds to 1.7e-18, which is floating-point arithmetic.

E4 is met in its first clause and not met in its second, and the way the second fails is the notebook's main finding about what a factor objective buys on industries. The four targeted factors' contributions sum to a positive amount on every main path: +5.7 to +16.6 basis points a year at 75, +26.7 to +35.5 at 100, +49.9 to +58.9 at 150, each with a standard error of 8 to 21 basis points, so the sum is beyond two standard errors on 10 of the 15 paths, the five at 100 (the monthly covariance's at 2.08 standard errors) and the five at 150, and on none at 75. The sum is the factors' realised returns over these 566 months times the exposures the budget bought, which is what the targets were chosen to collect; it estimates no premium and says nothing about the next 566 months. The factor share, defined in notebook 13 as one minus the variance of the residual over the variance of the active return, lies between 0.14 and 0.28 on the ten main paths at 100 and 150 basis points, where the design expected at least 0.30, and between 0.14 and 0.21 at 75: no path reaches the level, and the range is version 1's (0.14 to 0.25 under the mandate), unchanged by exposures three to seven times the size of version 1's incidental ones (the mean factor objective at 100 and 150 basis points against the sum of the four incidental exposures' absolute values, by estimator). The arithmetic says why. The six contributions' own variance is 34% to 52% of the active return's, above the level on every path at 100 and 150; the residual's variance is 72% to 86% of it; and the two move against each other, their covariance taking 18% to 26% off the sum, so that one minus the residual's part is 14% to 28%. The industries' own returns, the part the six factors do not describe, are the larger part of a factor objective's active return on 49 industries, and they move against the factor part: when the factors pay, the industries that carry the exposures give some of it back. The level of 0.30 was written from version 1's shares and the sentence that deliberate exposures raise the share; they raised the factor part's variance and the residual's with it, because the same active weights carry both.

The active return itself, the number a client is paid, is +6.0 to +15.5 basis points a year gross at 100 basis points on the five estimators, on standard errors of 17 to 22, and +20.5 to +34.0 at 150 on standard errors of 25 to 28; the largest ratio of an active return to its standard error across the nineteen paths is 1.36 (shrinkage at 150), so no path's active return is beyond two standard errors of zero. The residual is negative on every one of the nineteen paths, minus 10 to minus 47 basis points a year: the factor part earned +37 to +47 at 100 basis points and the industries' own returns took back 31 to 37 of it. Notebook 13 found the mirror image under version 1's mandate, a factor part of minus 11 to minus 17 and a residual of +7 to +14 on an active return near zero; the objective turned both parts round and left the sum inside its standard error. The two factors outside the objective are not at zero either. The active exposure to the market factor is 0.012 to 0.015 at 100 basis points on the four daily estimators and 0.003 on the monthly covariance, where version 1's mandate held it within 0.0015 of zero on the daily estimators, and it contributed +9.8 to +11.7 basis points a year on standard errors of 3.5 to 4.2 (the monthly covariance minus 0.5): beta neutrality is to the benchmark, the cap-weighted 49, and the industries the objective buys move more with French's market factor than the ones it sells. Size contributed +1.2 to +2.8 at 100. The targeted total, the quantity E4 asks about, excludes both.

The exposure bought, factor by factor, is what the budget was spent on. At 100 basis points on the five estimators the total is 0.00092 to 0.00096 of beta per basis point of realised tracking error, and it splits unevenly: profitability takes 0.00038 to 0.00040 and investment 0.00030 to 0.00042, the two of them 74% to 85% of the total, value 0.00012 to 0.00018 and momentum 0.00001 to 0.00007. The four were weighted equally in the objective; the optimiser bought most of the two whose industry betas it could raise cheaply within the mandate's bounds and least of momentum, whose betas are smallest and rotate fastest, so an equal-weighted target in beta units is not an equal-weighted purchase. The monthly covariance's path buys the most investment and the least momentum (0.00001), and the two single-factor paths show the cost of a single target: value alone buys 0.00060 of value and gives up 0.00030 of investment, momentum alone buys 0.00032 of momentum and gives up value and investment, so that its total exposure to the four is 0.00002 per basis point, a twentieth of the four-factor paths'.

The gap per unit of exposure bought is 0.039 to 0.044 at 100 basis points on the five estimators, 0.034 to 0.037 at 150 and 0.054 to 0.076 at 75: the betas estimated after the month take back 4% of the exposure chosen before it at the main budget, and a larger fraction at the small budget, where the exposure bought is small and the gap, which comes from the betas' movement and not from the budget, is not. Momentum alone has a smaller gap than value alone in beta units, 0.00185 against 0.00235, which is why E6's gap clause was not met in notebook 15, and a larger one per unit, 0.047 against 0.032, because it bought half the exposure; the design's sentence about momentum betas rotating faster holds per unit of what was bought and not in beta units. The largest single-month gaps fall in 1990-10 on eleven of the nineteen paths, in 2005-07 on five, and in 1981-09 and 1987-10 on the others, months in which the three-year beta windows turned over a shock.

By decade, at 100 basis points, the targeted total was negative in 1979 to 1989 on four of the five estimators (minus 6 to minus 18 basis points a year) and largest in the 1990s (+70 to +93), when the residual was also at its most negative (minus 104 to minus 143): the decade in which the factors paid most is the decade in which the industries carrying the exposures gave most of it back, and the active return of the decade is between minus 17 and +14. In the 2010s the factor part lies within 7 basis points of zero on the five estimators and the residual between minus 3 and minus 17. The by-decade table is reported as a description of where the contributions came from, with no claim attached to any decade.

**What this notebook does not settle.**

- The factor share has more than one definition. Defined as the six contributions' own variance over the active return's, it would be 0.34 to 0.52 and above the level; defined as notebook 13 defined it, one minus the residual's part, it is 0.14 to 0.28. The definition was written first and the verdict follows it; the decomposition into the two variances and their covariance is reported so that a reader can judge either.
- The attribution uses the betas estimated before each month, as the optimiser did, so a contribution is the exposure the manager chose times the factor's return. An attribution on the betas estimated after the month would move each contribution by the gap of that month; the gap table says by how much on average.
- The exposure bought and the gap depend on the betas' window, three years of daily returns, fixed before the code; a longer window would change both, and by how much is not measured here.
- Whether the targeted total's sign would hold over another sample is not a question these 566 months answer; the sum is the realised factor returns times the exposures, and the study estimates no premium.